| | |
|---|---|
| **CURSO** | Herramientas de Big Data |
| **SABER 2** | Procesamiento distribuido |
| **NOTEBOOK** | Informes y organización de entregables |
| **ACTIVIDAD** | Actividades 1 y 2: documentación, reportes y entregables |
| **DATASET** | Llamadas de urgencias y emergencias de la Línea 123 de Bogotá (Secretaría Distrital de Salud), julio–septiembre de 2026: 3 CSV, 27 434 registros, ~3 MB |
| **TIEMPO ESTIMADO** | ~6 minutos en Colab (runtime CPU) |

**Objetivos de aprendizaje:**
- Generar los informes (xlsx con fórmulas y docx en formato APA 7) a partir de las mediciones.
- Verificar fórmulas y número de páginas con LibreOffice dentro de Colab.
- Dejar en Drive una única carpeta con lo que pide la guía.

**Prerrequisitos:**
- U2_procesamiento_distribuido.ipynb ejecutado completo (mediciones en BigData_U2/resultados/).
- Datos de autoría del bloque de título completados en la celda de configuración (sección documentos).

# Informes y entregables — Emergencias 123 Bogotá (U2)

Este notebook **no vuelve a medir nada**: lee lo que dejó `U2_procesamiento_distribuido.ipynb` en `BigData_U2/resultados/` y genera
los informes. Después los verifica y organiza la carpeta única del proyecto en Drive:

| Carpeta en `MyDrive/BigData_U2/` | Contenido generado aquí |
|---|---|
| `Actividad_1/` | `reporte_comparativo_benchmark.xlsx` (mediciones, promedios, LOC, reglas y **matriz ponderada con fórmulas**), `analisis_decision_tecnologica.docx`, gráficos del benchmark |
| `Actividad_2/` | `arquitectura_tecnica.docx` (5–8 páginas), `diagrama_arquitectura.png` / `.drawio`, `metricas_poc.xlsx`, `comparativo_poc_vs_pandas.png`, `analisis_escalabilidad.docx` (1 página), `validacion_poc_spark.txt` |
| `Sustentacion/` | `Sustentacion_proyecto_U2.docx`: explicación para no especialistas |
| raíz | `LEEME.md`: qué archivo cumple cada tarea de la guía y cómo ejecutar |

Los informes se generan primero en una carpeta temporal del VM y luego se copian a su actividad; así en Drive no
quedan copias duplicadas ni archivos intermedios.

## 0. Entorno de ejecución

In [ ]:
# Entorno: Google Colab + Google Drive (única herramienta del proyecto)
import os, sys
try:
    import google.colab  # noqa: F401
except ImportError as e:
    raise RuntimeError('Este notebook se ejecuta en Google Colab.') from e

if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')  # Colab pide autorizar el acceso a Drive

BASE = '/content/drive/MyDrive/BigData_U2'   # carpeta ÚNICA del proyecto en Drive (datos, informes, evidencia)
LIB = '/content/u2lib'                         # código de apoyo que escribe este notebook (disco del VM; no se sube)
os.makedirs(BASE, exist_ok=True)
for carpeta in ('config', 'src/emergencias123/reportes', 'tests'):
    os.makedirs(f'{LIB}/{carpeta}', exist_ok=True)
sys.path.insert(0, f'{LIB}/src')
os.environ['U2_ENTORNO'] = 'colab'
print('Carpeta del proyecto en Drive:', BASE)

## Anexo A · Código de apoyo (ejecutar primero)

Estas celdas **escriben en el disco del VM** (`/content/u2lib`) el código de apoyo que usa el resto del notebook: la
configuración, las reglas de normalización, las operaciones de cada herramienta, el pipeline, las validaciones y
las pruebas. Cada archivo lleva antes una nota de qué hace. **No hace falta leerlas para seguir el relato**: el
código que importa se muestra en pantalla en las secciones donde se explica. Solo hay que ejecutarlas antes (la
opción *Ejecutar todas* ya lo hace). Nada de esto se sube a Drive.

**Configuración (`config/config.yaml`).** Toda ruta y todo parámetro sale de aquí; el código no tiene rutas fijas.
Contiene las rutas de Drive, los 3 archivos con su periodo, su **huella SHA-256** y la URL del portal (respaldo
heredado de la Unidad 1), el esquema observado en la inspección, los parámetros del benchmark y del PoC, la
configuración de Spark y los datos del bloque de título APA de los informes.

In [ ]:
%%writefile /content/u2lib/config/config.yaml
# Configuración del proyecto U2 — Emergencias 123 Bogotá.
# Las rutas relativas se resuelven contra la raíz del código (carpeta que contiene config/).
# El entorno se detecta solo (Colab si existe /content/drive/MyDrive); se puede forzar con U2_ENTORNO.

entornos:
  colab:                          # flujo principal: todo ocurre en Colab; una sola carpeta en Drive (BigData_U2)
    base: /content/drive/MyDrive/BigData_U2
    datos: /content/drive/MyDrive/BigData_U2/datos
    resultados: /content/drive/MyDrive/BigData_U2/resultados
    informes: /content/u2_informes   # temporal en el VM: el notebook de informes copia cada archivo a su Actividad_N/
    datos_alternos:               # si faltan CSV en `datos`: carpeta de Drive de la Unidad 1 (guía U1, paso 2)
      - /content/drive/MyDrive/BigData_Emergencias_123
  local:                          # solo para desarrollo y pruebas del código
    base: .
    datos: BigData_Emergencias_123
    resultados: resultados
    informes: informes

datos:
  separador: ";"
  codificacion: utf-8-sig        # los CSV traen BOM UTF-8
  # Fuente: Secretaría Distrital de Salud (SDS), portal Datos Abiertos Bogotá, licencia CC BY 4.0.
  # sha256 = huella de los archivos usados (idénticos a los del portal, verificado el 2026-11-10); `url` es el
  # respaldo de descarga que ya usaba la Unidad 1 cuando Drive no tiene los CSV.
  conjunto_url: https://datosabiertos.bogota.gov.co/dataset/llamadas-de-urgencias-y-emergencias-que-ingresan-a-traves-de-la-linea-123
  archivos:                      # periodo = mes que publica cada archivo
    - nombre: llamadas123_julio_2026.csv
      periodo: "2026-07"
      sha256: 21cbe766616d324086a26d91b337cbc3bf89e42585f2ffd0efb218d574b27589
      url: https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/e2b98357-3305-4058-a113-982490ab6b7e/download/llamadas123_julio_2026.csv
    - nombre: llamadaslinea123agosto2026.csv
      periodo: "2026-08"
      sha256: c421fe9d75edc4ed6da2afa153fe6c51e6576f371b4d140046aec57b66e5b2e1
      url: https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/a79acb9e-57ae-4e6c-bc15-3a8687330b1f/download/llamadaslinea123agosto2026.csv
    - nombre: llamadaslinea123Septiembre.csv
      periodo: "2026-09"
      sha256: face37df8c67912796511f9e6392d2a12dc139c91360eac1dfa46490aedd32d2
      url: https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/39f0e478-08f0-405c-b807-8015e07baec1/download/llamadaslinea123.csv

esquema:
  columnas:
    - NUMERO_INCIDENTE
    - FECHA_INICIO_DESPLAZAMIENTO_MOVIL
    - CODIGO_LOCALIDAD
    - LOCALIDAD
    - EDAD
    - UNIDAD
    - GENERO
    - TIPO_INCIDENTE
    - PRIORIDAD_FINAL
    - RECEPCION
  criticas:                      # un nulo aquí => el registro se conserva con observación de calidad
    - NUMERO_INCIDENTE
    - FECHA_INICIO_DESPLAZAMIENTO_MOVIL
    - CODIGO_LOCALIDAD
    - LOCALIDAD
    - TIPO_INCIDENTE
  valores_nulos: ["N/A", ""]     # marcadores de ausencia observados en la inspección
  localidad_sin_dato: SIN_D
  formato_fecha: "%Y-%m-%d %H:%M:%S"
  formato_fecha_spark: "yyyy-MM-dd HH:mm:ss"

catalogos:
  tipos: config/catalogo_tipos_derivado.csv          # derivado de los datos, NO oficial
  localidades: config/localidades_bogota.csv         # división político-administrativa de Bogotá

benchmark:
  herramientas: [pandas, polars, spark]
  modos: [por_operacion, extremo_a_extremo, extremo_a_extremo_caliente]   # el último es complementario
  localidades_filtro: [USAQUEN, CHAPINERO]           # valores ya normalizados (sin tildes)
  calentamientos: 1
  repeticiones: 5
  intervalo_muestreo_s: 0.02

poc:
  ejecuciones: 3
  tolerancia_dias_fin_mes: 1     # registros del día 1 del mes siguiente siguen en la ventana del archivo
  edad_min_anios: 0
  edad_max_anios: 120
  particionamiento: periodo      # periodo | periodo_localidad
  factores_edad:                 # conversión de EDAD a años según UNIDAD (normalizada)
    ANOS: 1.0
    MESES: 0.0833333333
    DIAS: 0.0027378508
    HORAS: 0.0001140771

documentos:                      # datos del bloque de título APA (completar antes de entregar)
  autor: "[Nombre del estudiante o del equipo]"
  institucion: "[Nombre de la institución]"
  curso: "Herramientas de Big Data — Saber 2: Procesamiento distribuido"
  docente: "[Nombre del docente]"
  fuente_datos: "Secretaría Distrital de Salud de Bogotá"           # autor (organización publicadora en el portal)
  titulo_datos: "Llamadas de urgencias y emergencias que ingresan a través de la línea 123"
  anio_datos: "2026"                                                # recursos jul–sep publicados en 2026
  editor_datos: "Datos Abiertos Bogotá"

spark:
  app_name: PoC_Emergencias123_Bogota
  driver_memory_max_gb: 4        # valor de la guía; se limita al 50 % de la RAM detectada
  driver_max_result_size: 2g
  advisory_partition_size: 64MB
  shuffle_partitions_por_cpu: 2
  zona_horaria: America/Bogota

Paquete `emergencias123`, que agrupa el código de apoyo de los notebooks.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/__init__.py
"""Proyecto U2 — Emergencias 123 Bogotá: benchmark Pandas/Polars/PySpark y PoC en PySpark."""

__version__ = '1.0.0'

Subpaquete `reportes` (gráficos e informes).

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/__init__.py
"""Generación de gráficos e informes a partir de las mediciones guardadas."""

**Lectura de la configuración (`config.py`).** Carga el YAML, resuelve las rutas del entorno de Colab y ofrece
utilidades (`rutas_csv`, `dir_salida`). Las variables `U2_DIR_DATOS` y `U2_DIR_RESULTADOS` permiten que el
benchmark trabaje sobre una copia en el disco local del VM sin cambiar el código.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/config.py
"""Carga de config/config.yaml y resolución de rutas según el entorno (local o Colab)."""
import os
from dataclasses import dataclass
from pathlib import Path

import yaml

RAIZ_CODIGO = Path(__file__).resolve().parents[2]
RUTA_CONFIG = RAIZ_CODIGO / 'config' / 'config.yaml'


def detectar_entorno():
    forzado = os.environ.get('U2_ENTORNO')
    if forzado:
        return forzado
    return 'colab' if Path('/content/drive/MyDrive').exists() else 'local'


@dataclass(frozen=True)
class Config:
    datos: dict
    esquema: dict
    catalogos: dict
    benchmark: dict
    poc: dict
    spark: dict
    documentos: dict
    entorno: str
    dir_datos: Path
    dir_resultados: Path
    dir_informes: Path
    dir_base: Path
    datos_alternos: tuple = ()     # otras carpetas donde buscar los CSV (p. ej. la de Drive de la Unidad 1)

    def ruta_csv(self, nombre):
        return self.dir_datos / nombre

    @property
    def rutas_csv(self):
        return [self.ruta_csv(a['nombre']) for a in self.datos['archivos']]

    @property
    def periodo_por_archivo(self):
        return {a['nombre']: a['periodo'] for a in self.datos['archivos']}

    @property
    def ruta_catalogo_tipos(self):
        return RAIZ_CODIGO / self.catalogos['tipos']

    @property
    def ruta_localidades(self):
        return RAIZ_CODIGO / self.catalogos['localidades']

    def dir_salida(self, *partes):
        ruta = self.dir_resultados.joinpath(*partes)
        ruta.mkdir(parents=True, exist_ok=True)
        return ruta


def _resolver(ruta):
    ruta = Path(ruta)
    return ruta if ruta.is_absolute() else RAIZ_CODIGO / ruta


def cargar_config(ruta=None, entorno=None, dir_resultados=None):
    """Lee el YAML. `dir_resultados` permite aislar salidas (pruebas o ejecuciones)."""
    with open(ruta or RUTA_CONFIG, encoding='utf-8') as f:
        crudo = yaml.safe_load(f)
    entorno = entorno or detectar_entorno()
    rutas = crudo['entornos'][entorno]
    return Config(
        datos=crudo['datos'], esquema=crudo['esquema'], catalogos=crudo['catalogos'],
        benchmark=crudo['benchmark'], poc=crudo['poc'], spark=crudo['spark'], documentos=crudo.get('documentos', {}),
        entorno=entorno,
        dir_datos=_resolver(os.environ.get('U2_DIR_DATOS', rutas['datos'])),
        dir_resultados=_resolver(dir_resultados or os.environ.get('U2_DIR_RESULTADOS', rutas['resultados'])),
        dir_informes=_resolver(rutas['informes']),
        dir_base=_resolver(rutas['base']),
        datos_alternos=tuple(_resolver(r) for r in rutas.get('datos_alternos', [])),
    )

**Gráficos (`graficos.py`)**: barras con desviación estándar a 300 dpi.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/graficos.py
"""Gráficos del benchmark y del PoC (matplotlib, 300 dpi).

Paleta categórica validada (orden fijo, color ligado a la herramienta y no a su posición):
Pandas azul, Polars naranja, Spark aqua. El aqua queda bajo 3:1 de contraste sobre la superficie,
por eso cada barra lleva su valor rotulado. Nunca se usan dos ejes Y: cada magnitud va en su panel.
"""
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

COLORES = {'pandas': '#2a78d6', 'polars': '#eb6834', 'spark': '#1baf7a'}
NOMBRES = {'pandas': 'Pandas', 'polars': 'Polars', 'spark': 'PySpark'}
ORDEN = ['pandas', 'polars', 'spark']
SUPERFICIE, TINTA, TINTA_2, MUTED, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
OPERACIONES = ['carga', 'filtrado', 'agregacion', 'join', 'exportacion']
ETIQUETAS_OP = {'carga': 'Carga', 'filtrado': 'Filtrado', 'agregacion': 'Agregación', 'join': 'Join',
                'exportacion': 'Exportación'}


def _estilo(ax, titulo, eje):
    ax.set_facecolor(SUPERFICIE)
    ax.set_title(titulo, loc='left', fontsize=11, color=TINTA, fontweight='bold', pad=10)
    ax.set_xlabel(eje, color=TINTA_2, fontsize=9)
    ax.tick_params(colors=TINTA_2, labelsize=9, length=0)
    for lado in ('top', 'right', 'left'):
        ax.spines[lado].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.grid(axis='x', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def _rotular(ax, barras, valores, desv, fmt):
    for b, v, e in zip(barras, valores, desv):
        ax.annotate(fmt.format(v), (b.get_width() + (e or 0), b.get_y() + b.get_height() / 2), xytext=(6, 0),
                    textcoords='offset points', va='center', fontsize=8, color=TINTA)


def _barras_h(ax, etiquetas, medias, desv, colores, fmt):
    pos = range(len(etiquetas))
    barras = ax.barh(list(pos), medias, xerr=desv, color=colores, height=0.6,
                     error_kw={'ecolor': MUTED, 'elinewidth': 1, 'capsize': 3})
    ax.set_yticks(list(pos), etiquetas)
    ax.invert_yaxis()
    _rotular(ax, barras, medias, desv, fmt)
    limite = max(m + (d or 0) for m, d in zip(medias, desv)) * 1.45
    ax.set_xlim(0, min(limite, 100) if '%' in fmt else limite)


def _figura(ncols, ancho, alto, nrows=1):
    fig, axes = plt.subplots(nrows, ncols, figsize=(ancho, alto), facecolor=SUPERFICIE, squeeze=False)
    return fig, list(axes.flat)


def _guardar(fig, ruta, titulo, nota):
    fig.suptitle(titulo, x=0.01, ha='left', fontsize=13, fontweight='bold', color=TINTA)
    fig.text(0.01, 0.005, nota, fontsize=7.5, color=TINTA_2, ha='left', va='bottom', wrap=True)
    fig.tight_layout(rect=(0, 0.06, 1, 0.94))
    fig.savefig(ruta, dpi=300, facecolor=SUPERFICIE)
    plt.close(fig)
    return ruta


def grafico_operaciones(medidas, ruta):
    """Tiempo medio por operación (modo por_operacion), un panel por operación (escalas propias)."""
    d = medidas[(medidas['modo'] == 'por_operacion') & medidas['operacion'].isin(OPERACIONES)]
    est = d.groupby(['operacion', 'herramienta'])['segundos'].agg(['mean', 'std']).reset_index()
    fig, axes = _figura(len(OPERACIONES), 17, 3.4)
    for ax, op in zip(axes, OPERACIONES):
        e = est[est['operacion'] == op].set_index('herramienta').reindex(ORDEN).dropna(subset=['mean'])
        _barras_h(ax, [NOMBRES[h] for h in e.index], e['mean'].tolist(), e['std'].fillna(0).tolist(),
                  [COLORES[h] for h in e.index], '{:.3f} s')
        _estilo(ax, ETIQUETAS_OP[op], 'segundos (media ± desv. est.)')
    n = int(d.groupby('herramienta')['ejecucion'].nunique().min())
    return _guardar(fig, ruta, 'Benchmark: tiempo por operación (modo por operación)',
                    f'{n} repeticiones medidas por herramienta, cada una en un proceso nuevo; calentamiento excluido. '
                    'Spark materializa cada operación con cache()+count(); el arranque de la sesión se mide aparte.')


def grafico_totales(medidas, ruta, por_fila=None):
    """Tiempo total por modo y memoria pico (RSS proceso+hijos y % del sistema), en paneles separados.

    `por_fila` reparte los paneles en varias filas (versión para documentos de página vertical).
    """
    tot = medidas[medidas['operacion'] == 'total']
    paneles = [
        ('Total extremo a extremo', tot[tot['modo'] == 'extremo_a_extremo'], 'segundos', 'segundos (media ± d.e.)', '{:.2f} s'),
        ('Total por operación (suma)', tot[tot['modo'] == 'por_operacion'], 'segundos', 'segundos (media ± d.e.)', '{:.2f} s'),
        ('Extremo a extremo, proceso caliente*', tot[tot['modo'] == 'extremo_a_extremo_caliente'], 'segundos',
         'segundos (media ± d.e.)', '{:.2f} s'),
        ('RSS pico proceso + hijos (JVM)', tot[tot['modo'] == 'extremo_a_extremo'], 'rss_pico_mb', 'MB (media ± d.e.)', '{:.0f} MB'),
        ('Memoria del sistema, pico', tot[tot['modo'] == 'extremo_a_extremo'], 'mem_sis_pico_pct', '% de la RAM total (media ± d.e.)', '{:.1f} %'),
    ]
    paneles = [p for p in paneles if len(p[1])]
    columnas = min(por_fila or len(paneles), len(paneles))
    filas = -(-len(paneles) // columnas)
    fig, axes = _figura(columnas, 4.3 * columnas, 3.2 * filas + 0.4, nrows=filas)
    for ax in axes[len(paneles):]:
        ax.set_visible(False)
    for ax, (titulo, d, col, eje, fmt) in zip(axes, paneles):
        e = d.groupby('herramienta')[col].agg(['mean', 'std']).reindex(ORDEN).dropna(subset=['mean'])
        _barras_h(ax, [NOMBRES[h] for h in e.index], e['mean'].tolist(), e['std'].fillna(0).tolist(),
                  [COLORES[h] for h in e.index], fmt)
        _estilo(ax, titulo, eje)
    return _guardar(fig, ruta, 'Benchmark: tiempo total y memoria (de la carga a la exportación)',
                    'Tiempos sin el arranque de SparkSession. *Proceso caliente: 2.ª pasada en el mismo proceso tras una '
                    'pasada sin medir (complementario). Memoria muestreada cada 20 ms; el % del sistema incluye todo el '
                    'runtime de Colab y no es memoria exclusiva de la herramienta.')


def grafico_poc(metricas, ruta):
    """Comparación PoC PySpark vs Pandas (mismo alcance): tiempo, memoria del sistema y RSS."""
    paneles = [
        ('Tiempo total carga→exportación', 'tiempo_total_s', 'segundos (media ± d.e.)', '{:.2f} s'),
        ('Memoria del sistema, pico', 'mem_sis_pico_pct', '% de la RAM total (media ± d.e.)', '{:.1f} %'),
        ('RSS pico proceso + hijos (JVM)', 'rss_pico_mb', 'MB (media ± d.e.)', '{:.0f} MB'),
    ]
    fig, axes = _figura(3, 14, 3.0)
    for ax, (titulo, col, eje, fmt) in zip(axes, paneles):
        e = metricas.groupby('herramienta')[col].agg(['mean', 'std']).reindex(['pandas', 'spark']).dropna(subset=['mean'])
        _barras_h(ax, [NOMBRES[h] for h in e.index], e['mean'].tolist(), e['std'].fillna(0).tolist(),
                  [COLORES[h] for h in e.index], fmt)
        _estilo(ax, titulo, eje)
    n = int(metricas.groupby('herramienta')['ejecucion'].nunique().min())
    return _guardar(fig, ruta, 'PoC: PySpark frente a Pandas con el mismo alcance (3 meses, Colab)',
                    f'{n} ejecuciones por herramienta, cada una en un proceso nuevo. Valores más bajos son mejores. '
                    'El tiempo de Spark excluye el arranque de la sesión, que se reporta aparte.')


def cargar_mediciones(ruta_jsonl):
    df = pd.read_json(ruta_jsonl, lines=True)
    return df[(df['estado'] == 'ok') & (~df['calentamiento'].astype(bool))] if 'calentamiento' in df else df

**Cifras de los informes (`resumen.py`).** Todas salen de `resultados/mediciones/`: ningún número de los
documentos se escribe a mano.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/resumen.py
"""Cifras que alimentan los documentos, calculadas SOLO a partir de resultados/mediciones/.

También contiene el modelo de escalabilidad: ajuste lineal tiempo/memoria vs. registros sobre los
puntos MEDIDOS (3 meses reales y replicados x4/x8). Lo que exceda esos puntos es extrapolación teórica.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd

HERRAMIENTAS = ['pandas', 'polars', 'spark']
REGISTROS_MES = 27434 / 3


def _leer_jsonl(ruta):
    return pd.read_json(ruta, lines=True) if Path(ruta).exists() else pd.DataFrame()


def resumen_benchmark(dir_med):
    crudo = _leer_jsonl(dir_med / 'benchmark.jsonl')
    errores = int((crudo['estado'] != 'ok').sum())
    med = crudo[(crudo['estado'] == 'ok') & (~crudo['calentamiento'].astype(bool))]
    tot = (med[med['operacion'] == 'total'].groupby(['modo', 'herramienta'])
           .agg(seg=('segundos', 'mean'), seg_de=('segundos', 'std'), seg_min=('segundos', 'min'), seg_max=('segundos', 'max'),
                rss=('rss_pico_mb', 'mean'), sis=('mem_sis_pico_pct', 'mean'), sis_base=('mem_sis_base_pct', 'mean'),
                n=('segundos', 'size')))
    ops = (med[(med['modo'] == 'por_operacion')].groupby(['operacion', 'herramienta'])['segundos'].mean().unstack())
    arranque = med[(med['herramienta'] == 'spark') & (med['operacion'] == 'inicio_sesion')]['segundos']
    eq_ruta = dir_med / 'equivalencia_benchmark.json'
    equivalencia = json.loads(eq_ruta.read_text(encoding='utf-8')) if eq_ruta.exists() else {}
    return {'med': med, 'crudo': crudo, 'errores': errores, 'tot': tot, 'ops': ops,
            'arranque_spark': float(arranque.mean()), 'arranque_spark_de': float(arranque.std()),
            'repeticiones': int(med.groupby(['herramienta', 'modo'])['ejecucion'].nunique().min()),
            'equivalencia': equivalencia,
            'equivalentes': all(r.get('equivalentes') for r in equivalencia.values()) if equivalencia else None}


def resumen_poc(dir_med):
    crudo = _leer_jsonl(dir_med / 'poc.jsonl')
    ok = crudo[crudo['estado'] == 'ok']
    principal = ok[ok['etiqueta'].isna()]
    variante = ok[ok['etiqueta'] == 'variante_periodo_localidad']
    cols = ['tiempo_total_s', 'tiempo_inicio_sesion_s', 'mem_sis_pico_pct', 'mem_sis_base_pct', 'rss_pico_mb',
            'throughput_reg_min', 'tasa_validaciones_pct', 'registros_validos_pct', 'leidos', 'aceptados',
            'rechazados', 'errores', 'parquet_archivos', 'parquet_bytes']
    prom = principal.groupby('herramienta')[cols].mean()
    de = principal.groupby('herramienta')[cols].std()
    ultima = principal[principal['herramienta'] == 'spark'].sort_values('ejecucion').iloc[-1]
    return {'crudo': crudo, 'principal': principal, 'variante': variante, 'prom': prom, 'de': de,
            'ultima_spark': ultima, 'errores_ejecucion': int((crudo['estado'] != 'ok').sum()),
            'ejecuciones': int(principal.groupby('herramienta')['ejecucion'].nunique().min())}


def modelo_escalabilidad(poc, dir_med):
    """Ajuste lineal por herramienta: y = a + b·N con los puntos medidos (factor 1 real; 4 y 8 replicados)."""
    esc = _leer_jsonl(dir_med / 'escalabilidad.jsonl')
    if esc.empty:
        return None
    esc = esc[esc['estado'] == 'ok']
    puntos = []
    for h in ('spark', 'pandas'):
        p = poc['prom'].loc[h]
        puntos.append({'herramienta': h, 'factor': 1, 'leidos': p['leidos'], 'tiempo_total_s': p['tiempo_total_s'],
                       'rss_pico_mb': p['rss_pico_mb'], 'mem_sis_pico_pct': p['mem_sis_pico_pct'], 'origen': 'medido (real)'})
        for _, r in esc[esc['herramienta'] == h].iterrows():
            puntos.append({'herramienta': h, 'factor': int(r['factor_replica']), 'leidos': r['leidos'],
                           'tiempo_total_s': r['tiempo_total_s'], 'rss_pico_mb': r['rss_pico_mb'],
                           'mem_sis_pico_pct': r['mem_sis_pico_pct'], 'origen': 'medido (replicado)'})
    puntos = pd.DataFrame(puntos).sort_values(['herramienta', 'factor'])
    ajustes = {}
    for h, d in puntos.groupby('herramienta'):
        bt, at = np.polyfit(d['leidos'], d['tiempo_total_s'], 1)
        bm, am = np.polyfit(d['leidos'], d['rss_pico_mb'], 1)
        ajustes[h] = {'t_fijo_s': at, 't_por_millon_s': bt * 1e6, 'rss_fijo_mb': am, 'rss_por_millon_mb': bm * 1e6}
    return {'puntos': puntos, 'ajustes': ajustes}


def proyectar(ajuste, registros):
    return (ajuste['t_fijo_s'] + ajuste['t_por_millon_s'] * registros / 1e6,
            ajuste['rss_fijo_mb'] + ajuste['rss_por_millon_mb'] * registros / 1e6)


def cargar_todo(dir_resultados):
    dir_med = Path(dir_resultados) / 'mediciones'
    entorno = json.loads((dir_med / 'entorno_colab.json').read_text(encoding='utf-8'))
    inventario = json.loads((dir_med / 'inventario_datos.json').read_text(encoding='utf-8'))
    bench = resumen_benchmark(dir_med)
    poc = resumen_poc(dir_med)
    ruta_optim = dir_med / 'optimizacion_poc.json'
    return {'entorno': entorno, 'inventario': inventario, 'bench': bench, 'poc': poc,
            'escala': modelo_escalabilidad(poc, dir_med),
            'optim': json.loads(ruta_optim.read_text(encoding='utf-8')) if ruta_optim.exists() else None}

**Diagrama de arquitectura (`diagrama.py`).** Una sola especificación de nodos y flechas produce la fuente
editable `.drawio` y el PNG a 300 dpi.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/diagrama.py
"""Diagrama de arquitectura del PoC desde UNA especificación: fuente editable .drawio + PNG de alta resolución.

Las coordenadas están en unidades de draw.io (px); el PNG se dibuja con matplotlib usando las mismas
coordenadas, así ambas salidas representan exactamente lo mismo.
"""
from xml.sax.saxutils import escape

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch  # noqa: E402

ANCHO, ALTO = 1700, 980
CARRILES = [  # id, título, x, ancho, color de fondo
    ('ingesta', '1 · Ingesta', 20, 330, '#EEF4FB'),
    ('proc', '2 · Procesamiento (PySpark 4.0, local[2] en Colab)', 370, 520, '#EEF8F3'),
    ('alm', '3 · Almacenamiento (Google Drive)', 910, 360, '#FDF3EC'),
    ('sal', '4 · Salidas, monitoreo y KPIs', 1290, 390, '#F4F2FB'),
]
COLOR = {'dato': '#2a78d6', 'proc': '#1a8a62', 'alm': '#c4561f', 'sal': '#6250d6', 'conf': '#7a7873'}
NODOS = [  # id, x, y, ancho, alto, tipo, texto
    ('fuente', 40, 90, 290, 90, 'dato', 'Línea 123 Bogotá\n3 CSV mensuales (jul–sep 2026)\n27 434 registros · ~3 MB · «;» · UTF-8'),
    ('drive_in', 40, 230, 290, 80, 'alm', 'Google Drive\nMyDrive/BigData_U2/datos\n(huella SHA-256 verificada)'),
    ('montaje', 40, 360, 290, 80, 'dato', 'Colab VM (2 vCPU, 12.7 GB)\nDrive montado en /content/drive'),
    ('config', 40, 490, 290, 120, 'conf', 'Configuración versionada\nconfig.yaml · esquema · periodos\nlocalidades_bogota.csv (oficial)\ncatalogo_tipos_derivado.csv\n(derivado, NO oficial)'),
    ('codigo', 40, 660, 290, 80, 'conf', 'Código: 2 notebooks de Colab\n(MyDrive/BigData_U2/*.ipynb)'),
    ('prev', 390, 90, 230, 80, 'proc', 'Prevalidación\narchivos · columnas · filas'),
    ('lectura', 640, 90, 230, 80, 'proc', 'Lectura CSV\nesquema explícito (texto)\n+ _corrupt_record'),
    ('limpieza', 390, 200, 230, 80, 'proc', 'Nulos explícitos (N/A → null)\nsin imputación · procedencia\n(archivo_origen, periodo)'),
    ('tipos', 640, 200, 230, 80, 'proc', 'Tipado seguro (ANSI)\ntry_to_timestamp · try_cast\nedad en años según UNIDAD'),
    ('estd', 390, 320, 230, 80, 'proc', 'Estandarización\nlocalidad · código de tipo\nprioridad · género'),
    ('reglas', 640, 320, 230, 80, 'proc', 'Reglas de calidad\nanomalías marcadas (se conservan)\nduplicados exactos eliminados'),
    ('join', 390, 440, 230, 80, 'proc', 'Broadcast join\ncatálogo de tipos (23 filas)\nm:1, sin multiplicar filas'),
    ('agg', 640, 440, 230, 80, 'proc', 'Agregados\nperiodo×localidad×tipo\nperiodo×prioridad · diario'),
    ('escritura', 390, 560, 480, 70, 'proc', 'Escritura Parquet particionada por periodo\nrepartition(periodo) · sortWithinPartitions(localidad)'),
    ('valid', 390, 670, 480, 80, 'proc', 'Validación de salidas V01–V20 (relectura de Parquet)\ntasa = aprobados / evaluados · conservados / leídos'),
    ('det', 930, 90, 320, 90, 'alm', 'parquet/poc/spark/eN/detalle/\nperiodo=2026-07 · 2026-08 · 2026-09\n(22 columnas tipadas)'),
    ('aggp', 930, 230, 320, 80, 'alm', 'parquet/poc/spark/eN/agregados/\n3 tablas resumidas'),
    ('rech', 930, 360, 320, 80, 'alm', 'parquet/poc/spark/eN/duplicados_eliminados/\ncopias exactas (100 % de atributos)'),
    ('med', 930, 490, 320, 80, 'alm', 'resultados/mediciones/\nbenchmark.jsonl · poc.jsonl · entorno'),
    ('logs', 930, 620, 320, 70, 'alm', 'resultados/logs/\nvalidacion_poc_spark.txt'),
    ('monitor', 1310, 90, 350, 110, 'sal', 'Monitoreo (MemorySampler, 20 ms)\n% RAM del sistema (psutil)\nRSS proceso + JVM · tiempo perf_counter'),
    ('kpi', 1310, 250, 350, 110, 'sal', 'KPIs del PoC\ntiempo total (s) · memoria pico (%)\nthroughput (registros/min)\ntasa de validaciones (%)'),
    ('informes', 1310, 410, 350, 100, 'sal', 'Informes\nxlsx (fórmulas) · docx · gráficos PNG\nnotebooks ejecutados en Colab'),
    ('consumo', 1310, 560, 350, 100, 'sal', 'Consumo analítico\nconsultas por periodo/localidad\n(poda de particiones + estadísticas)'),
]
ARISTAS = [  # origen, destino, etiqueta
    ('fuente', 'drive_in', 'carga'), ('drive_in', 'montaje', 'FUSE'), ('montaje', 'prev', ''),
    ('config', 'prev', ''), ('codigo', 'valid', ''),
    ('prev', 'lectura', ''), ('lectura', 'limpieza', ''), ('limpieza', 'tipos', ''), ('tipos', 'estd', ''),
    ('estd', 'reglas', ''), ('reglas', 'join', 'conservados'), ('join', 'agg', ''), ('agg', 'escritura', ''),
    ('escritura', 'valid', ''), ('escritura', 'det', ''), ('escritura', 'aggp', ''), ('reglas', 'rech', 'duplicados'),
    ('valid', 'logs', ''), ('valid', 'med', 'métricas'), ('med', 'kpi', ''), ('logs', 'kpi', ''),
    ('med', 'informes', ''), ('det', 'consumo', ''), ('monitor', 'kpi', ''),
]
NOTA = ('Ejecución real implementada: Spark en modo local dentro de UNA VM de Colab (no es un clúster). '
        'Despliegue distribuido propuesto: mismo código en Databricks / clúster Spark con almacenamiento de objetos.')


def _nodo(nid):
    return next(n for n in NODOS if n[0] == nid)


def _anclas(a, b):
    """Puntos de salida/entrada entre dos cajas (lado más cercano)."""
    _, ax, ay, aw, ah, *_ = a
    _, bx, by, bw, bh, *_ = b
    if bx >= ax + aw:  # destino a la derecha
        return (ax + aw, ay + ah / 2), (bx, by + bh / 2)
    if bx + bw <= ax:  # destino a la izquierda
        return (ax, ay + ah / 2), (bx + bw, by + bh / 2)
    if by >= ay + ah:  # destino abajo
        return (ax + aw / 2, ay + ah), (bx + bw / 2, by)
    return (ax + aw / 2, ay), (bx + bw / 2, by + bh)


def escribir_drawio(ruta):
    celdas = ['<mxCell id="0"/>', '<mxCell id="1" parent="0"/>']
    for cid, titulo, x, w, fondo in CARRILES:
        celdas.append(f'<mxCell id="carril_{cid}" value="{escape(titulo)}" style="swimlane;startSize=36;fillColor={fondo};'
                      f'strokeColor=#BFBFBF;fontStyle=1;fontSize=15;rounded=1;" vertex="1" parent="1">'
                      f'<mxGeometry x="{x}" y="20" width="{w}" height="{ALTO - 110}" as="geometry"/></mxCell>')
    for nid, x, y, w, h, tipo, texto in NODOS:
        valor = escape(texto).replace('\n', '&#xa;')
        celdas.append(f'<mxCell id="{nid}" value="{valor}" style="rounded=1;whiteSpace=wrap;html=0;fillColor=#FFFFFF;'
                      f'strokeColor={COLOR[tipo]};strokeWidth=2;fontSize=12;" vertex="1" parent="1">'
                      f'<mxGeometry x="{x}" y="{y + 20}" width="{w}" height="{h}" as="geometry"/></mxCell>')
    for i, (o, d, etiqueta) in enumerate(ARISTAS):
        celdas.append(f'<mxCell id="e{i}" value="{escape(etiqueta)}" style="endArrow=block;html=0;strokeColor=#52514e;'
                      f'fontSize=11;edgeStyle=orthogonalEdgeStyle;rounded=1;" edge="1" parent="1" source="{o}" target="{d}">'
                      f'<mxGeometry relative="1" as="geometry"/></mxCell>')
    celdas.append(f'<mxCell id="nota" value="{escape(NOTA)}" style="text;html=0;fontSize=12;fontStyle=2;align=left;" '
                  f'vertex="1" parent="1"><mxGeometry x="20" y="{ALTO - 80}" width="{ANCHO - 40}" height="40" as="geometry"/></mxCell>')
    xml = ('<?xml version="1.0" encoding="UTF-8"?>\n<mxfile host="emergencias123" type="device">'
           '<diagram id="arquitectura" name="Arquitectura PoC Emergencias 123">'
           f'<mxGraphModel dx="1600" dy="900" grid="1" gridSize="10" page="1" pageWidth="{ANCHO}" pageHeight="{ALTO}">'
           f'<root>{"".join(celdas)}</root></mxGraphModel></diagram></mxfile>\n')
    ruta.write_text(xml, encoding='utf-8')
    return ruta


def escribir_png(ruta, dpi=300):
    fig = plt.figure(figsize=(ANCHO / 100, ALTO / 100), facecolor='#FCFCFB')
    ax = fig.add_axes((0, 0, 1, 1))
    ax.set_xlim(0, ANCHO)
    ax.set_ylim(ALTO, 0)
    ax.axis('off')
    ax.text(20, 8, 'Arquitectura del PoC — Pipeline PySpark de Emergencias 123 Bogotá (U2)', fontsize=17,
            fontweight='bold', va='top', color='#0b0b0b')
    for _, titulo, x, w, fondo in CARRILES:
        ax.add_patch(FancyBboxPatch((x, 40), w, ALTO - 130, boxstyle='round,pad=0,rounding_size=12',
                                    facecolor=fondo, edgecolor='#BFBFBF', linewidth=1))
        ax.text(x + 12, 62, titulo, fontsize=12.5, fontweight='bold', va='center', color='#0b0b0b')
    for nid, x, y, w, h, tipo, texto in NODOS:
        ax.add_patch(FancyBboxPatch((x, y + 20), w, h, boxstyle='round,pad=0,rounding_size=10',
                                    facecolor='#FFFFFF', edgecolor=COLOR[tipo], linewidth=2))
        lineas = texto.split('\n')
        paso = 15.5
        y0 = y + 20 + h / 2 - (len(lineas) - 1) * paso / 2
        for k, linea in enumerate(lineas):
            ax.text(x + w / 2, y0 + k * paso, linea, ha='center', va='center', fontsize=9.0,
                    color='#0b0b0b' if k == 0 else '#3a3936', fontweight='bold' if k == 0 else 'normal')
    for o, d, etiqueta in ARISTAS:
        (x1, y1), (x2, y2) = _anclas(_nodo(o), _nodo(d))
        ax.add_patch(FancyArrowPatch((x1, y1 + 20), (x2, y2 + 20), arrowstyle='-|>', mutation_scale=13,
                                     color='#52514e', linewidth=1.3, connectionstyle='arc3,rad=0.0', shrinkA=2, shrinkB=2))
        if etiqueta:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 14, etiqueta, fontsize=8.5, color='#52514e', ha='center',
                    bbox={'facecolor': '#FCFCFB', 'edgecolor': 'none', 'pad': 1})
    leyenda = [('Datos/infraestructura', 'dato'), ('Procesamiento Spark', 'proc'), ('Almacenamiento', 'alm'),
               ('Salidas/monitoreo', 'sal'), ('Configuración', 'conf')]
    for i, (nombre, tipo) in enumerate(leyenda):
        x = 20 + i * 210
        ax.add_patch(FancyBboxPatch((x, ALTO - 60), 18, 14, boxstyle='round,pad=0,rounding_size=3',
                                    facecolor='#FFFFFF', edgecolor=COLOR[tipo], linewidth=2))
        ax.text(x + 26, ALTO - 53, nombre, fontsize=9.5, va='center', color='#52514e')
    ax.text(20, ALTO - 22, NOTA, fontsize=10, style='italic', color='#52514e', va='center')
    fig.savefig(ruta, dpi=dpi, facecolor='#FCFCFB')
    plt.close(fig)
    return ruta

**Libros Excel (`excel.py`).** Promedios, desviaciones, puntajes y la matriz ponderada se calculan con
**fórmulas** (`AVERAGE`, `STDEV`, `SUMPRODUCT`), no con valores pegados.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/excel.py
"""Reportes .xlsx con fórmulas (promedios, desviaciones, puntajes y ponderaciones se calculan en Excel).

Las mediciones crudas se escriben como valores. Todo lo derivado son fórmulas que las referencian,
para que el libro se recalcule si cambian los datos. La desviación estándar muestral se calcula con
SUMPRODUCT (sin fórmulas matriciales) para que funcione en cualquier Excel.
"""
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

FUENTE = 'Arial'
AZUL_INPUT = Font(name=FUENTE, color='0000FF')
NEGRITA = Font(name=FUENTE, bold=True)
NORMAL = Font(name=FUENTE)
TITULO = Font(name=FUENTE, bold=True, size=14)
CABECERA_FILL = PatternFill('solid', fgColor='DCE6F1')
INPUT_FILL = PatternFill('solid', fgColor='FFF2CC')
BORDE = Border(bottom=Side(style='thin', color='BFBFBF'))
HERRAMIENTAS = ['pandas', 'polars', 'spark']
NOMBRES = {'pandas': 'Pandas', 'polars': 'Polars', 'spark': 'PySpark'}
OPERACIONES = ['inicio_sesion', 'carga', 'filtrado', 'agregacion', 'join', 'exportacion', 'extremo_a_extremo',
               'total', 'total_con_arranque']
MODOS = ['por_operacion', 'extremo_a_extremo', 'extremo_a_extremo_caliente']


def _cabecera(ws, fila, columnas, anchos=None):
    for j, c in enumerate(columnas, 1):
        celda = ws.cell(fila, j, c)
        celda.font, celda.fill, celda.border = NEGRITA, CABECERA_FILL, BORDE
        celda.alignment = Alignment(wrap_text=True, vertical='center')
    for j, ancho in enumerate(anchos or [], 1):
        ws.column_dimensions[get_column_letter(j)].width = ancho


def _fuente_por_defecto(wb):
    for ws in wb.worksheets:
        for fila in ws.iter_rows():
            for c in fila:
                if c.value is not None and (c.font is None or c.font.name != FUENTE):
                    c.font = Font(name=FUENTE, bold=c.font.bold, color=c.font.color, size=c.font.size)


def _texto(ws, fila, texto, fuente=NORMAL, col=1):
    c = ws.cell(fila, col, texto)
    c.font, c.alignment = fuente, Alignment(wrap_text=True, vertical='top')
    return c


def _escribir_mediciones(ws, med, columnas):
    _cabecera(ws, 1, columnas, [12, 26, 9, 13, 18, 11, 12, 12, 11, 11, 11, 11, 7, 9, 40, 8])
    for i, fila in enumerate(med[columnas].itertuples(index=False), 2):
        for j, v in enumerate(fila, 1):
            if hasattr(v, 'item'):
                v = v.item()
            ws.cell(i, j, None if v != v else v)  # NaN -> vacío
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(columnas))}{len(med) + 1}'
    return len(med) + 1


def _rangos(ultima, letras):
    return {k: f'Mediciones!${c}$2:${c}${ultima}' for k, c in letras.items()}


def reporte_benchmark(med, loc, equivalencia, entorno, ruta):
    """med: todas las filas de benchmark.jsonl (incluye calentamientos, que se excluyen por fórmula)."""
    wb = Workbook()
    ws0 = wb.active
    ws0.title = 'Resumen'
    columnas = ['herramienta', 'modo', 'ejecucion', 'calentamiento', 'operacion', 'segundos', 'mem_sis_base_pct',
                'mem_sis_pico_pct', 'rss_base_mb', 'rss_pico_mb', 'filas_entrada', 'filas_salida', 'loc',
                'usa_cache', 'accion_materializacion', 'estado']
    wsm = wb.create_sheet('Mediciones')
    ultima = _escribir_mediciones(wsm, med, columnas)
    R = _rangos(ultima, {'h': 'A', 'modo': 'B', 'cal': 'D', 'op': 'E', 'seg': 'F', 'sis': 'H', 'rss': 'J'})

    # --- Promedios (fórmulas sobre Mediciones, calentamiento excluido) ---
    wsp = wb.create_sheet('Promedios')
    _cabecera(wsp, 1, ['Modo', 'Operación', 'Herramienta', 'n', 'Segundos (media)', 'Segundos (desv. est.)',
                       'Segundos (mín.)', 'Segundos (máx.)', 'RSS pico MB (media)', 'Memoria sistema pico % (media)',
                       'Veces vs. la más rápida'], [22, 18, 12, 6, 14, 14, 13, 13, 14, 16, 14])
    fila, filas_total = 2, {}
    presentes = set(zip(med['modo'], med['operacion'], med['herramienta']))
    for modo in MODOS:
        for op in OPERACIONES:
            bloque = [h for h in HERRAMIENTAS if (modo, op, h) in presentes]
            if not bloque:
                continue
            inicio = fila
            for h in bloque:
                crit = f'{R["h"]},$C{fila},{R["modo"]},$A{fila},{R["op"]},$B{fila},{R["cal"]},FALSE'
                wsp.cell(fila, 1, modo)
                wsp.cell(fila, 2, op)
                wsp.cell(fila, 3, h)
                wsp.cell(fila, 4, f'=COUNTIFS({crit})')
                wsp.cell(fila, 5, f'=IFERROR(AVERAGEIFS({R["seg"]},{crit}),"")')
                cond = (f'({R["h"]}=$C{fila})*({R["modo"]}=$A{fila})*({R["op"]}=$B{fila})*({R["cal"]}=FALSE)')
                wsp.cell(fila, 6, f'=IF($D{fila}>1,SQRT(SUMPRODUCT({cond}*({R["seg"]}-$E{fila})^2)/($D{fila}-1)),"")')
                wsp.cell(fila, 7, f'=_xlfn.MINIFS({R["seg"]},{crit})')
                wsp.cell(fila, 8, f'=_xlfn.MAXIFS({R["seg"]},{crit})')
                wsp.cell(fila, 9, f'=IFERROR(AVERAGEIFS({R["rss"]},{crit}),"")')
                wsp.cell(fila, 10, f'=IFERROR(AVERAGEIFS({R["sis"]},{crit}),"")')
                if op == 'total':
                    filas_total[(modo, h)] = fila
                fila += 1
            for f in range(inicio, fila):
                wsp.cell(f, 11, f'=IFERROR(E{f}/MIN($E${inicio}:$E${fila - 1}),"")')
                for col, fmt in ((5, '0.000'), (6, '0.000'), (7, '0.000'), (8, '0.000'), (9, '0.0'), (10, '0.0'), (11, '0.0"x"')):
                    wsp.cell(f, col).number_format = fmt
    wsp.freeze_panes = 'A2'

    # --- LOC ---
    wsl = wb.create_sheet('LOC')
    ops5 = ['carga', 'filtrado', 'agregacion', 'join', 'exportacion']
    _cabecera(wsl, 1, ['Operación'] + [NOMBRES[h] for h in HERRAMIENTAS], [18, 12, 12, 12])
    for i, op in enumerate(ops5, 2):
        wsl.cell(i, 1, op)
        for j, h in enumerate(HERRAMIENTAS, 2):
            wsl.cell(i, j, loc[h][op])
    fila_tot_loc = len(ops5) + 2
    wsl.cell(fila_tot_loc, 1, 'TOTAL').font = NEGRITA
    for j in range(2, 5):
        col = get_column_letter(j)
        wsl.cell(fila_tot_loc, j, f'=SUM({col}2:{col}{fila_tot_loc - 1})').font = NEGRITA
    _texto(wsl, fila_tot_loc + 2, 'Criterio: líneas físicas no vacías ni de comentario entre los marcadores '
           '"# >>> op:<nombre>" y "# <<< op:<nombre>" de src/emergencias123/ops_*.py (sin la firma def).')

    # --- Reglas de puntuación ---
    wsr = wb.create_sheet('Reglas')
    wsr.column_dimensions['A'].width = 24
    wsr.column_dimensions['B'].width = 110
    reglas = [
        ('Escala', 'Puntajes de 1 (peor) a 5 (mejor). Puntaje ponderado = SUMPRODUCT(pesos, puntajes).'),
        ('Cuantitativos', 'r = valor de la herramienta / mejor valor entre las tres. r<=1.25 -> 5; r<=2 -> 4; r<=4 -> 3; '
                          'r<=8 -> 2; r>8 -> 1. Calculado por fórmula en la hoja Matriz.'),
        ('Rendimiento (30 %)', 'Tiempo total extremo a extremo (proceso nuevo por repetición, sin arranque de sesión), '
                               'media de las repeticiones medidas. Fuente: hoja Promedios.'),
        ('Recursos (15 %)', 'Pico de RSS del proceso más sus hijos (incluye la JVM), media, modo extremo a extremo. '
                            'Se usa RSS y no el % del sistema porque este último incluye procesos ajenos.'),
        ('Facilidad de uso (25 %)', 'Base cuantitativa: cociente de LOC totales (misma escala r). Ajuste cualitativo '
                                    'de -1 a +1 justificado en la hoja Matriz (configuración, depuración, '
                                    'dependencias). Resultado acotado a [1, 5].'),
        ('Escalabilidad (20 %)', 'Cualitativo: 5 = distribuido multinodo nativo; 3 = un nodo multihilo con ejecución '
                                 'streaming/out-of-core; 2 = un nodo, en memoria, principalmente un hilo.'),
        ('Curva de aprendizaje (10 %)', 'Cualitativo: 5 = API ampliamente conocida y sin conceptos nuevos; 4 = API '
                                        'nueva de expresiones similar a SQL/dataframes; 2 = conceptos de motor '
                                        'distribuido (evaluación perezosa, DAG, particiones, JVM, configuración).'),
        ('Celdas editables', 'Las celdas con fondo amarillo y texto azul son juicios cualitativos o ajustes '
                             'editables; todo lo demás se calcula.'),
    ]
    for i, (a, b) in enumerate(reglas, 1):
        _texto(wsr, i, a, NEGRITA)
        _texto(wsr, i, b, col=2)

    # --- Matriz de decisión ---
    wsx = wb.create_sheet('Matriz')
    for col, ancho in zip('ABCDEFGH', [26, 9, 14, 14, 14, 3, 70, 3]):
        wsx.column_dimensions[col].width = ancho
    _texto(wsx, 1, 'Matriz de decisión tecnológica ponderada (escala 1-5)', TITULO)
    _texto(wsx, 3, 'Insumos cuantitativos (fórmulas sobre Promedios y LOC)', NEGRITA)
    _cabecera(wsx, 4, ['Insumo', '', 'Pandas', 'Polars', 'PySpark'])
    t = filas_total
    insumos = [
        ('Tiempo total extremo a extremo (s)', [f"=Promedios!E{t[('extremo_a_extremo', h)]}" for h in HERRAMIENTAS], '0.000'),
        ('RSS pico proceso+hijos (MB)', [f"=Promedios!I{t[('extremo_a_extremo', h)]}" for h in HERRAMIENTAS], '0.0'),
        ('LOC totales', [f'=LOC!{c}{fila_tot_loc}' for c in 'BCD'], '0'),
    ]
    for i, (nombre, formulas, fmt) in enumerate(insumos, 5):
        wsx.cell(i, 1, nombre)
        for j, f in enumerate(formulas, 3):
            wsx.cell(i, j, f).number_format = fmt

    def puntaje(fila_insumo, col):
        r = f'{col}{fila_insumo}/MIN($C${fila_insumo}:$E${fila_insumo})'
        return f'=IF({r}<=1.25,5,IF({r}<=2,4,IF({r}<=4,3,IF({r}<=8,2,1))))'

    _texto(wsx, 9, 'Ajuste cualitativo de facilidad de uso (-1, 0, +1)', NEGRITA)
    ajustes = {'pandas': 0, 'polars': 0, 'spark': -1}
    for j, h in enumerate(HERRAMIENTAS, 3):
        c = wsx.cell(9, j, ajustes[h])
        c.font, c.fill = AZUL_INPUT, INPUT_FILL
    _texto(wsx, 9, 'PySpark -1: requiere configurar SparkSession y JVM, sus errores mezclan trazas de Java y Python y la '
           'evaluación perezosa exige acciones explícitas para ver resultados. Pandas y Polars se usan y depuran como una '
           'biblioteca más de Python.', col=7)

    _cabecera(wsx, 11, ['Criterio', 'Peso', 'Pandas', 'Polars', 'PySpark', '', 'Fundamento'])
    criterios = [
        ('Rendimiento', 0.30, [puntaje(5, c) for c in 'CDE'],
         'Cuantitativo: cociente del tiempo total extremo a extremo frente a la herramienta más rápida (fila 5).'),
        ('Facilidad de uso', 0.25, [f'=MAX(1,MIN(5,{puntaje(7, c)[1:]}+{c}9))' for c in 'CDE'],
         'Cociente de LOC (fila 7) más el ajuste cualitativo de la fila 9.'),
        ('Escalabilidad', 0.20, [2, 3, 5],
         'Pandas: un nodo, todo en memoria. Polars: un nodo, multihilo, streaming/out-of-core. '
         'PySpark: escala horizontalmente a un clúster (Databricks, Dataproc, etc.) sin cambiar el código.'),
        ('Recursos', 0.15, [puntaje(6, c) for c in 'CDE'],
         'Cuantitativo: cociente del pico de RSS del proceso más sus hijos (fila 6; incluye la JVM).'),
        ('Curva de aprendizaje', 0.10, [5, 4, 2],
         'Pandas: estándar de facto y la más conocida. Polars: API de expresiones nueva pero familiar. '
         'PySpark: exige evaluación perezosa, DAG, particiones, caché y configuración de la JVM.'),
    ]
    for i, (nombre, peso, puntajes, fundamento) in enumerate(criterios, 12):
        wsx.cell(i, 1, nombre)
        c = wsx.cell(i, 2, peso)
        c.number_format, c.font, c.fill = '0%', AZUL_INPUT, INPUT_FILL
        for j, p in enumerate(puntajes, 3):
            c = wsx.cell(i, j, p)
            if not str(p).startswith('='):
                c.font, c.fill = AZUL_INPUT, INPUT_FILL
        _texto(wsx, i, fundamento, col=7)
    fp = 12 + len(criterios)
    wsx.cell(fp, 1, 'Suma de pesos').font = NEGRITA
    wsx.cell(fp, 2, f'=SUM(B12:B{fp - 1})').number_format = '0%'
    wsx.cell(fp + 1, 1, 'Puntaje ponderado').font = NEGRITA
    for col in 'CDE':
        c = wsx.cell(fp + 1, ' CDE'.index(col) + 2, f'=SUMPRODUCT($B$12:$B${fp - 1},{col}12:{col}{fp - 1})')
        c.number_format, c.font = '0.00', NEGRITA
    wsx.cell(fp + 2, 1, 'Posición (1 = mejor)').font = NEGRITA
    for col in 'CDE':
        wsx.cell(fp + 2, ' CDE'.index(col) + 2, f'=RANK({col}{fp + 1},$C${fp + 1}:$E${fp + 1})')
    wsx.cell(fp + 3, 1, 'Mayor puntaje').font = NEGRITA
    wsx.cell(fp + 3, 3, f'=INDEX($C$4:$E$4,MATCH(MAX($C${fp + 1}:$E${fp + 1}),$C${fp + 1}:$E${fp + 1},0))')
    _texto(wsx, fp + 5, 'Lectura: la matriz refleja el volumen probado (3 meses, ~3 MB). PySpark es la herramienta '
           'principal OBLIGATORIA de la actividad; si otra herramienta obtiene mayor puntaje, eso indica cuál es más '
           'conveniente para este volumen, no reemplaza la elección exigida. Ver el análisis de decisión (.docx).', col=1)
    wsx.merge_cells(start_row=fp + 5, start_column=1, end_row=fp + 5, end_column=7)
    wsx.row_dimensions[fp + 5].height = 48

    # --- Gráficos nativos ---
    wsg = wb.create_sheet('Graficos')
    _cabecera(wsg, 1, ['Herramienta', 'Total extremo a extremo (s)', 'Total por operación (s)',
                       'Extremo a extremo, proceso caliente (s)', 'RSS pico (MB)', 'Memoria sistema pico (%)'],
              [14, 18, 18, 22, 14, 16])
    for i, h in enumerate(HERRAMIENTAS, 2):
        wsg.cell(i, 1, NOMBRES[h])
        for j, (modo, col) in enumerate((('extremo_a_extremo', 'E'), ('por_operacion', 'E'),
                                         ('extremo_a_extremo_caliente', 'E'), ('extremo_a_extremo', 'I'),
                                         ('extremo_a_extremo', 'J')), 2):
            if (modo, h) in t:
                wsg.cell(i, j, f'=Promedios!{col}{t[(modo, h)]}').number_format = '0.000'
    for k, (titulo, col, eje, ancla) in enumerate((
            ('Tiempo total de la carga a la exportación (media, s)', 2, 'segundos', 'A7'),
            ('RSS pico proceso + hijos (media, MB)', 5, 'MB', 'J7'),
            ('Memoria del sistema, pico (media, %)', 6, '% RAM', 'A24'))):
        g = BarChart()
        g.type, g.title, g.y_axis.title, g.x_axis.title = 'bar', titulo, eje, 'Herramienta'
        g.add_data(Reference(wsg, min_col=col, max_col=col + (2 if k == 0 else 0), min_row=1, max_row=4), titles_from_data=True)
        g.set_categories(Reference(wsg, min_col=1, min_row=2, max_row=4))
        g.height, g.width = 7.5, 16
        wsg.add_chart(g, ancla)

    # --- Equivalencia ---
    wse = wb.create_sheet('Equivalencia')
    _cabecera(wse, 1, ['Modo', 'Criterio', 'Resultado', 'Detalle'], [24, 14, 12, 90])
    i = 2
    for modo, r in equivalencia.items():
        for criterio, ok in r['criterios'].items():
            detalle = {h: (x[criterio] if criterio in ('filas',) else (x[criterio][:16] if criterio == 'sha256' else ''))
                       for h, x in r['resumenes'].items()}
            wse.cell(i, 1, modo)
            wse.cell(i, 2, criterio)
            wse.cell(i, 3, 'IGUAL' if ok else 'DIFERENTE')
            wse.cell(i, 4, str(detalle) if any(detalle.values()) else 'idéntico en las 3 herramientas' if ok else '')
            i += 1

    # --- Resumen ---
    ws0.column_dimensions['A'].width = 120
    lineas = [
        ('Reporte comparativo del benchmark Pandas vs Polars vs PySpark — Emergencias 123 Bogotá (U2)', TITULO),
        ('Fuente de datos: resultados/mediciones/benchmark.jsonl (ejecución real en Google Colab; '
         f"{entorno.get('fecha', '')}). Hoja Mediciones = valores crudos; el resto son fórmulas.", NORMAL),
        (f"Entorno: Python {entorno.get('python')}, pandas {entorno.get('pandas')}, polars {entorno.get('polars')}, "
         f"pyspark {entorno.get('pyspark')}, {entorno.get('java')}, {entorno.get('vcpu')} vCPU, "
         f"{entorno.get('ram_gb')} GB RAM. Spark en modo local (un nodo), no clúster.", NORMAL),
        ('Datos: 3 CSV mensuales (jul-sep 2026), 27 434 registros. Filtro USAQUEN+CHAPINERO = 1 762 registros. '
         'Catálogo de tipos DERIVADO de los datos (no oficial).', NORMAL),
        ('Metodología: measure_performance(); proceso nuevo por repetición; 1 calentamiento descartado + 5 medidas; '
         'orden rotado; reloj perf_counter_ns; memoria muestreada cada 20 ms (pico = máximo de muestras).', NORMAL),
        ('Modos: por_operacion (cada operación materializada; Spark con cache()+count()), extremo_a_extremo '
         '(mínima materialización) y extremo_a_extremo_caliente (complementario: 2.ª pasada en el mismo proceso).', NORMAL),
        ('Hojas: Mediciones · Promedios · LOC · Reglas · Matriz · Graficos · Equivalencia.', NORMAL),
        ('Limitaciones: volumen pequeño (dominan costes fijos); % de memoria del sistema incluye otros procesos de Colab; '
         'RSS incluye páginas compartidas; muestreo de 20 ms puede perder picos muy breves.', NORMAL),
    ]
    for i, (texto, fuente) in enumerate(lineas, 1):
        _texto(ws0, i, texto, fuente)
    _fuente_por_defecto(wb)
    wb.save(ruta)
    return ruta


def reporte_poc(poc, controles, ruta, entorno):
    """poc: filas de poc.jsonl (estado ok). controles: lista de controles de la última ejecución de Spark."""
    wb = Workbook()
    ws0 = wb.active
    ws0.title = 'Resumen'
    principal = poc[poc['etiqueta'].isna()].sort_values(['herramienta', 'ejecucion'])
    cols = ['herramienta', 'ejecucion', 'ts_inicio', 'tiempo_total_s', 'tiempo_inicio_sesion_s', 'tiempo_verificacion_s',
            'mem_sis_base_pct', 'mem_sis_pico_pct', 'rss_base_mb', 'rss_pico_mb', 'leidos', 'aceptados', 'rechazados',
            'throughput_reg_min', 'tasa_validaciones_pct', 'registros_validos_pct', 'controles_aprobados',
            'controles_evaluados', 'errores', 'parquet_archivos', 'parquet_bytes']
    wse = wb.create_sheet('Ejecuciones')
    _cabecera(wse, 1, cols, [11, 9, 20] + [13] * (len(cols) - 3))
    for i, fila in enumerate(principal[cols].itertuples(index=False), 2):
        for j, v in enumerate(fila, 1):
            wse.cell(i, j, v.item() if hasattr(v, 'item') else v)
    n = len(principal) + 1
    # Throughput recalculado por fórmula a partir de leídos y tiempo (comprueba la métrica guardada)
    wse.cell(1, len(cols) + 1, 'throughput (fórmula) reg/min').font = NEGRITA
    for i in range(2, n + 1):
        wse.cell(i, len(cols) + 1, f'=K{i}/(D{i}/60)').number_format = '#,##0'

    wsp = wb.create_sheet('Promedios')
    metricas = [('Tiempo total carga→exportación (s)', 'D', '0.000'), ('Arranque de sesión (s, aparte)', 'E', '0.000'),
                ('Memoria sistema pico (%)', 'H', '0.0'), ('RSS pico proceso+JVM (MB)', 'J', '0.0'),
                ('Throughput (registros/min)', 'N', '#,##0'), ('Tasa de éxito de validaciones (%)', 'O', '0.00'),
                ('Registros conservados (%)', 'P', '0.0000'), ('Registros procesados (leídos)', 'K', '#,##0'),
                ('Errores (controles fallidos)', 'S', '0')]
    _cabecera(wsp, 1, ['Métrica', 'PySpark media', 'PySpark desv. est.', 'Pandas media', 'Pandas desv. est.',
                       'PySpark / Pandas'], [36, 14, 16, 14, 16, 16])
    for i, (nombre, col, fmt) in enumerate(metricas, 2):
        wsp.cell(i, 1, nombre)
        for k, h in enumerate(('spark', 'pandas')):
            rango, crit = f'Ejecuciones!${col}$2:${col}${n}', f'Ejecuciones!$A$2:$A${n},"{h}"'
            media = wsp.cell(i, 2 + 2 * k, f'=AVERAGEIFS({rango},{crit})')
            media.number_format = fmt
            ref_media = f'{get_column_letter(2 + 2 * k)}{i}'
            desv = wsp.cell(i, 3 + 2 * k, f'=IF(COUNTIFS({crit})>1,SQRT(SUMPRODUCT((Ejecuciones!$A$2:$A${n}="{h}")*'
                                          f'({rango}-{ref_media})^2)/(COUNTIFS({crit})-1)),"")')
            desv.number_format = fmt
        wsp.cell(i, 6, f'=IFERROR(B{i}/D{i},"n/a (división entre 0)")').number_format = '0.00"x"'
    _texto(wsp, len(metricas) + 3, 'PySpark / Pandas > 1 significa que PySpark obtiene un valor mayor (en tiempo y '
           'memoria, peor; en throughput, mejor). Las cifras provienen de 3 ejecuciones por herramienta en Colab, '
           'cada una en un proceso nuevo.')
    wsp.merge_cells(start_row=len(metricas) + 3, start_column=1, end_row=len(metricas) + 3, end_column=6)
    wsp.row_dimensions[len(metricas) + 3].height = 45

    g = BarChart()
    g.type, g.title, g.y_axis.title = 'bar', 'Tiempo total medio (s): PySpark vs Pandas', 'segundos'
    g.add_data(Reference(wsp, min_col=2, max_col=2, min_row=1, max_row=2), titles_from_data=True)
    g.add_data(Reference(wsp, min_col=4, max_col=4, min_row=1, max_row=2), titles_from_data=True)
    g.set_categories(Reference(wsp, min_col=1, min_row=2, max_row=2))
    g.height, g.width = 6.5, 15
    wsp.add_chart(g, 'H2')
    g2 = BarChart()
    g2.type, g2.title, g2.y_axis.title = 'bar', 'RSS pico medio (MB): PySpark vs Pandas', 'MB'
    g2.add_data(Reference(wsp, min_col=2, max_col=2, min_row=4, max_row=5), titles_from_data=False)
    g2.add_data(Reference(wsp, min_col=4, max_col=4, min_row=4, max_row=5), titles_from_data=False)
    g2.set_categories(Reference(wsp, min_col=1, min_row=5, max_row=5))
    g2.height, g2.width = 6.5, 15
    wsp.add_chart(g2, 'H17')

    wsv = wb.create_sheet('Validaciones')
    _cabecera(wsv, 1, ['Id', 'Severidad', 'Descripción', 'Valor', 'Esperado', 'Aprobado'], [6, 12, 60, 30, 22, 10])
    for i, c in enumerate(controles, 2):
        for j, k in enumerate(('id', 'severidad', 'descripcion', 'valor', 'esperado', 'aprobado'), 1):
            v = c[k]
            wsv.cell(i, j, v if isinstance(v, (int, float, bool)) else str(v))
    fin = len(controles) + 1
    wsv.cell(fin + 2, 3, 'Controles evaluados (bloqueante + advertencia)').font = NEGRITA
    wsv.cell(fin + 2, 6, f'=COUNTIF(B2:B{fin},"bloqueante")+COUNTIF(B2:B{fin},"advertencia")')
    wsv.cell(fin + 3, 3, 'Controles aprobados').font = NEGRITA
    wsv.cell(fin + 3, 6, f'=COUNTIFS(B2:B{fin},"bloqueante",F2:F{fin},TRUE)+COUNTIFS(B2:B{fin},"advertencia",F2:F{fin},TRUE)')
    wsv.cell(fin + 4, 3, 'Tasa de éxito de validaciones = aprobados / evaluados').font = NEGRITA
    wsv.cell(fin + 4, 6, f'=F{fin + 3}/F{fin + 2}').number_format = '0.00%'

    variante = poc[poc['etiqueta'] == 'variante_periodo_localidad']
    wsq = wb.create_sheet('Particionamiento')
    _cabecera(wsq, 1, ['Estrategia', 'Archivos Parquet', 'Bytes', 'KB por archivo', 'Tiempo total (s)'], [22, 16, 14, 14, 16])
    ultima_spark = principal[principal['herramienta'] == 'spark'].tail(1)
    for i, (nombre, d) in enumerate((('periodo', ultima_spark), ('periodo/localidad', variante)), 2):
        if len(d):
            r = d.iloc[0]
            wsq.cell(i, 1, nombre)
            wsq.cell(i, 2, int(r['parquet_archivos']))
            wsq.cell(i, 3, int(r['parquet_bytes']))
            wsq.cell(i, 4, f'=C{i}/B{i}/1024').number_format = '0.0'
            wsq.cell(i, 5, float(r['tiempo_total_s'])).number_format = '0.00'
    _texto(wsq, 5, 'La variante periodo/localidad es una sola ejecución de referencia (no entra en los promedios).')

    ws0.column_dimensions['A'].width = 120
    for i, (texto, fuente) in enumerate((
        ('Reporte de métricas del PoC PySpark (3 ejecuciones) y comparación con Pandas — Emergencias 123 (U2)', TITULO),
        (f"Fuente: resultados/mediciones/poc.jsonl (ejecución real en Google Colab, {entorno.get('fecha', '')}). "
         'Hoja Ejecuciones = valores crudos; Promedios y tasas = fórmulas.', NORMAL),
        ('Tiempo total = carga → exportación (sin arranque de la SparkSession, que se reporta aparte). Throughput = '
         'registros leídos / minuto. Memoria del sistema = pico muestreado de psutil.virtual_memory().percent; RSS = '
         'proceso + hijos (JVM).', NORMAL),
        ('Tasa de éxito de validaciones = controles aprobados / controles evaluados (bloqueantes + advertencias). '
         'Registros conservados = conservados / leídos; solo se eliminan las copias duplicadas exactas (columnas '
         '"aceptados" = conservados y "rechazados" = copias eliminadas). Son métricas distintas.', NORMAL),
        (f"Entorno: {entorno.get('vcpu')} vCPU, {entorno.get('ram_gb')} GB RAM, pyspark {entorno.get('pyspark')}, "
         f"pandas {entorno.get('pandas')}; Spark local (un nodo).", NORMAL),
    ), 1):
        _texto(ws0, i, texto, fuente)
    _fuente_por_defecto(wb)
    wb.save(ruta)
    return ruta

**Documentos Word en formato APA 7 (`documentos.py`).** Bloque de título, encabezados sin numeración, tablas y
figuras numeradas con nota, citas y referencias con sangría francesa.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/documentos.py
"""Documentos .docx (python-docx). Toda cifra proviene de `resumen.cargar_todo()` o de la matriz recalculada.

La redacción que depende de los resultados (qué herramienta gana, cuántas veces más lenta, etc.) se
construye a partir de las cifras, no se escribe a mano.
"""
import re

from docx import Document
from docx.enum.section import WD_ORIENT  # noqa: F401  (disponible si se requiere apaisado)
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt, RGBColor

from .resumen import REGISTROS_MES, proyectar

NOMBRE = {'pandas': 'Pandas', 'polars': 'Polars', 'spark': 'PySpark'}
GRIS = RGBColor(0x52, 0x51, 0x4E)
NEGRO = RGBColor(0, 0, 0)


class Doc:
    """Documento con formato APA 7 (adaptado a los límites de páginas de la guía).

    Bloque de título centrado, encabezados sin numeración (nivel 1 centrado, nivel 2 a la izquierda),
    tablas y figuras numeradas («Tabla N» / «Figura N» en negrita y título en cursiva, «Nota.» debajo),
    tablas solo con líneas horizontales, texto en negro y referencias con sangría francesa.
    """

    def __init__(self, tamano=11, margen_cm=2.54, interlineado=1.15):
        self.d = Document()
        for s in self.d.sections:
            s.top_margin = s.bottom_margin = s.left_margin = s.right_margin = Cm(margen_cm)
        normal = self.d.styles['Normal']
        normal.font.name, normal.font.size = 'Arial', Pt(tamano)
        normal.element.rPr.rFonts.set(qn('w:eastAsia'), 'Arial')
        normal.font.color.rgb = NEGRO
        normal.paragraph_format.space_after = Pt(4)
        normal.paragraph_format.line_spacing = interlineado
        for nivel, tam in (('Title', tamano + 3), ('Heading 1', tamano + 1), ('Heading 2', tamano)):
            st = self.d.styles[nivel]
            st.font.name, st.font.size, st.font.bold, st.font.italic = 'Arial', Pt(tam), True, False
            st.font.color.rgb = NEGRO
            fuentes = st.element.rPr.rFonts
            fuentes.set(qn('w:eastAsia'), 'Arial')
            for atributo in ('w:asciiTheme', 'w:hAnsiTheme', 'w:eastAsiaTheme', 'w:cstheme'):
                fuentes.attrib.pop(qn(atributo), None)  # la fuente del tema (Calibri) anularía Arial
            borde = st.element.pPr.find(qn('w:pBdr')) if st.element.pPr is not None else None
            if borde is not None:  # el estilo Title trae una línea inferior que APA no usa
                st.element.pPr.remove(borde)
            st.paragraph_format.space_before = Pt(10 if nivel != 'Title' else 0)
            st.paragraph_format.space_after = Pt(4)
            st.paragraph_format.keep_with_next = True
        self.tamano, self.n_tabla, self.n_figura = tamano, 0, 0

    def bloque_titulo(self, titulo, datos, compacto=False):
        """Título (negrita, centrado) y datos de autoría según APA; `datos` viene de config.yaml (documentos)."""
        t = self.d.add_paragraph(titulo, style='Title')
        t.alignment = WD_ALIGN_PARAGRAPH.CENTER
        lineas = [datos['autor'], datos['institucion'], datos['curso'], datos['docente'], datos['fecha']]
        for linea in ([' · '.join(lineas)] if compacto else lineas):
            par = self.d.add_paragraph(linea)
            par.alignment = WD_ALIGN_PARAGRAPH.CENTER
            par.paragraph_format.space_after = Pt(0)
        self.d.add_paragraph().paragraph_format.space_after = Pt(2 if compacto else 6)

    def h1(self, texto):
        h = self.d.add_heading(re.sub(r'^\d+(\.\d+)*\.?\s+', '', texto), level=1)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER

    def h2(self, texto):
        self.d.add_heading(re.sub(r'^\d+(\.\d+)*\.?\s+', '', texto), level=2)

    def p(self, texto, negrita_inicial=None):
        par = self.d.add_paragraph()
        if negrita_inicial:
            par.add_run(negrita_inicial).bold = True
        par.add_run(texto)
        par.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
        return par

    def vinetas(self, elementos):
        for e in elementos:
            par = self.d.add_paragraph(style='List Bullet')
            if isinstance(e, tuple):
                par.add_run(e[0]).bold = True
                par.add_run(e[1])
            else:
                par.add_run(e)
            par.paragraph_format.space_after = Pt(1)

    def nota(self, texto, tam=None):
        """Nota al estilo APA: «Nota.» en cursiva seguida del texto."""
        par = self.d.add_paragraph()
        r = par.add_run('Nota. ')
        r.italic, r.font.size = True, Pt(tam or self.tamano - 1.5)
        par.add_run(texto).font.size = Pt(tam or self.tamano - 1.5)
        par.paragraph_format.space_after = Pt(6)

    def _rotulo(self, tipo, numero, titulo):
        a = self.d.add_paragraph()
        a.add_run(f'{tipo} {numero}').bold = True
        a.paragraph_format.space_after = Pt(0)
        a.paragraph_format.keep_with_next = True
        b = self.d.add_paragraph()
        b.add_run(titulo).italic = True
        b.paragraph_format.space_after = Pt(2)
        b.paragraph_format.keep_with_next = True

    @staticmethod
    def _bordes_apa(tabla):
        """Solo líneas horizontales (superior, entre filas e inferior), como recomienda APA."""
        bordes = OxmlElement('w:tblBorders')
        for lado in ('top', 'bottom', 'insideH'):
            e = OxmlElement(f'w:{lado}')
            e.set(qn('w:val'), 'single')
            e.set(qn('w:sz'), '6')
            e.set(qn('w:color'), '000000')
            bordes.append(e)
        for lado in ('left', 'right', 'insideV'):
            e = OxmlElement(f'w:{lado}')
            e.set(qn('w:val'), 'nil')
            bordes.append(e)
        tabla._tbl.tblPr.append(bordes)

    def tabla(self, cabecera, filas, anchos_cm=None, tam=None, titulo=None, nota=None):
        if titulo:
            self.n_tabla += 1
            self._rotulo('Tabla', self.n_tabla, titulo)
        t = self.d.add_table(rows=1, cols=len(cabecera))
        t.alignment = WD_TABLE_ALIGNMENT.CENTER
        self._bordes_apa(t)
        tam = Pt(tam or self.tamano - 1.5)
        for c, texto in zip(t.rows[0].cells, cabecera):
            c.text = ''
            r = c.paragraphs[0].add_run(str(texto))
            r.bold, r.font.size = True, tam
        for fila in filas:
            celdas = t.add_row().cells
            for c, v in zip(celdas, fila):
                c.text = ''
                c.paragraphs[0].add_run(str(v)).font.size = tam
        if anchos_cm:
            for fila in t.rows:
                for c, a in zip(fila.cells, anchos_cm):
                    c.width = Cm(a)
        if nota:
            self.nota(nota, tam=tam.pt)
        else:
            self.d.add_paragraph().paragraph_format.space_after = Pt(2)
        return t

    def imagen(self, ruta, ancho_cm, titulo, nota=None):
        self.n_figura += 1
        self._rotulo('Figura', self.n_figura, titulo)
        self.d.add_picture(str(ruta), width=Cm(ancho_cm))
        self.d.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
        if nota:
            self.nota(nota)

    def referencias(self, entradas, tam=None):
        """Referencias APA: orden alfabético, sangría francesa de 1,27 cm; cada entrada = [(texto, cursiva), ...]."""
        self.h1('Referencias')
        for fragmentos in sorted(entradas, key=lambda e: ''.join(t for t, _ in e).lower()):
            par = self.d.add_paragraph()
            par.paragraph_format.left_indent = Cm(1.27)
            par.paragraph_format.first_line_indent = Cm(-1.27)
            par.paragraph_format.space_after = Pt(3)
            for texto, cursiva in fragmentos:
                r = par.add_run(texto)
                r.italic = cursiva
                if tam:
                    r.font.size = Pt(tam)

    def salto(self):
        self.d.add_page_break()

    def guardar(self, ruta):
        self.d.save(ruta)
        return ruta


def f(x, dec=2):
    return f'{x:,.{dec}f}'.replace(',', ' ')


def java_corto(texto):
    """'openjdk version "21.0.12.1" 2026-08-18' -> 'OpenJDK 21.0.12.1'."""
    m = re.search(r'"([\d._]+)"', texto)
    return f'OpenJDK {m.group(1)}' if m else texto


def veces(a, b):
    return a / b if b else float('nan')


def _datos_clave(R, matriz):
    b, poc = R['bench'], R['poc']
    tot = b['tot']
    e2e = tot.loc['extremo_a_extremo']
    pop = tot.loc['por_operacion']
    cal = tot.loc['extremo_a_extremo_caliente'] if 'extremo_a_extremo_caliente' in tot.index.get_level_values(0) else None
    rapida = e2e['seg'].idxmin()
    ganador = max(matriz['ponderado'], key=matriz['ponderado'].get)
    ps, pp = poc['prom'].loc['spark'], poc['prom'].loc['pandas']
    return dict(b=b, poc=poc, e2e=e2e, pop=pop, cal=cal, rapida=rapida, ganador=ganador, ps=ps, pp=pp,
                ent=R['entorno'], inv=R['inventario'])


# ---------------------------------------------------------------- Referencias APA 7 (fragmentos: texto, cursiva)
MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre',
         'noviembre', 'diciembre']


def fecha_es(iso):
    anio, mes, dia = int(iso[:4]), int(iso[5:7]), int(iso[8:10])
    return f'{dia} de {MESES[mes - 1]} de {anio}'


def referencias_apa(datos, ent):
    anio = ent['fecha'][:4]
    return {
        'guia': [(f"{datos['institucion']}. ({anio}). ", False),
                 ('Guía de actividades: Unidad 2. Procesamiento distribuido', True), (' [Guía de curso].', False)],
        'datos': [(f"{datos['fuente_datos']}. ({anio}). ", False),
                  ('Llamadas a la Línea 123 de Bogotá: julio, agosto y septiembre de 2026', True),
                  (' [Conjunto de datos suministrado para el curso].', False)],
        'spark': [('Apache Software Foundation. (2025). ', False), ('Apache Spark', True),
                  (f" (Versión {ent['pyspark']}) [Software]. https://spark.apache.org/", False)],
        'parquet': [('Apache Software Foundation. (s. f.). ', False), ('Apache Parquet', True),
                    (' [Especificación de formato de archivo]. https://parquet.apache.org/', False)],
        'colab': [('Google. (s. f.). ', False), ('Google Colaboratory', True),
                  (' [Plataforma en línea]. https://colab.research.google.com/', False)],
        'mckinney': [('McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt y '
                      'J. Millman (Eds.), ', False), ('Proceedings of the 9th Python in Science Conference', True),
                     (' (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a', False)],
        'pandas': [('The pandas development team. (s. f.). ', False), ('pandas-dev/pandas: Pandas', True),
                   (f" (Versión {ent['pandas']}) [Software]. Zenodo. https://doi.org/10.5281/zenodo.3509134", False)],
        'polars': [('Polars Developers. (s. f.). ', False), ('Polars', True),
                   (f" (Versión {ent['polars']}) [Software]. https://pola.rs/", False)],
        'psutil': [('Rodola, G. (s. f.). ', False), ('psutil', True),
                   (f" (Versión {ent['psutil']}) [Software]. https://github.com/giampaolo/psutil", False)],
        'zaharia': [('Zaharia, M., Xin, R. S., Wendell, P., Das, T., Armbrust, M., Dave, A., Meng, X., Rosen, J., '
                     'Venkataraman, S., Franklin, M. J., Ghodsi, A., Gonzalez, J., Shenker, S. y Stoica, I. (2016). '
                     'Apache Spark: A unified engine for big data processing. ', False),
                    ('Communications of the ACM, 59', True), ('(11), 56–65. https://doi.org/10.1145/2934664', False)],
    }


def _cita_guia(datos, ent):
    return f"({datos['institucion']}, {ent['fecha'][:4]})"


# ---------------------------------------------------------------- Análisis de decisión
def doc_decision(R, matriz, rutas, ruta, datos):
    k = _datos_clave(R, matriz)
    e2e, pop, cal, ent, ps = k['e2e'], k['pop'], k['cal'], k['ent'], k['ps']
    datos = {**datos, 'fecha': fecha_es(ent['fecha'])}
    refs = referencias_apa(datos, ent)
    d = Doc()
    d.bloque_titulo('Análisis de decisión tecnológica para el procesamiento de las llamadas a la Línea 123 de Bogotá',
                    datos)

    d.h1('Resumen ejecutivo')
    orden = sorted(matriz['ponderado'], key=matriz['ponderado'].get, reverse=True)
    d.p(f"Se compararon Pandas, Polars y PySpark con los tres archivos mensuales seleccionados "
        f"({f(R['inventario']['total_registros'], 0)} registros, aproximadamente 3 MB), ejecutados en Google Colab "
        f"(Google, s. f.). La herramienta más rápida de la carga a la exportación fue {NOMBRE[k['rapida']]}, con "
        f"{f(e2e.loc[k['rapida'], 'seg'], 3)} s de media. PySpark requirió {f(e2e.loc['spark', 'seg'], 2)} s, es decir, "
        f"{f(veces(e2e.loc['spark', 'seg'], e2e.loc[k['rapida'], 'seg']), 0)} veces más, sin contar los "
        f"{f(k['b']['arranque_spark'], 1)} s de arranque de la sesión. En memoria residente del proceso (RSS, por sus "
        f"siglas en inglés), PySpark alcanzó {f(e2e.loc['spark', 'rss'], 0)} MB, frente a "
        f"{f(e2e.loc['pandas', 'rss'], 0)} MB de Pandas y {f(e2e.loc['polars', 'rss'], 0)} MB de Polars.")
    d.p("La matriz ponderada ordena las herramientas así: " +
        ', '.join(f"{NOMBRE[h]} ({f(matriz['ponderado'][h], 2)})" for h in orden) +
        f". En consecuencia, {NOMBRE[k['ganador']]} es la opción más conveniente para el volumen probado. PySpark se "
        f"mantiene como herramienta principal porque la actividad lo exige {_cita_guia(datos, ent)} y porque es la "
        "única de las tres que escala horizontalmente a un clúster sin reescribir el código (Zaharia et al., 2016). "
        "Pandas y Polars se recomiendan como herramientas complementarias para la exploración y para los lotes que caben "
        "en la memoria de un solo nodo.")

    d.h1('Contexto, datos y método')
    d.vinetas([
        ('Datos. ', f"Tres archivos CSV de la Línea 123 (julio, agosto y septiembre de 2026; "
                    f"{datos['fuente_datos']}, {ent['fecha'][:4]}) con {f(R['inventario']['total_registros'], 0)} "
                    "registros, codificación UTF-8 con BOM y separador «;». Al no encontrarse una tabla oficial de "
                    "códigos de emergencia, la unión emplea un catálogo de 23 tipos derivado de TIPO_INCIDENTE, "
                    "identificado como no oficial."),
        ('Operaciones. ', 'Idénticas en las tres herramientas: carga y combinación; filtrado de las localidades USAQUÉN '
         'y CHAPINERO con el texto normalizado (1 762 registros); conteo por tipo; unión izquierda muchos a uno con el '
         'catálogo, y exportación a Parquet (Apache Software Foundation, s. f.).'),
        ('Entorno. ', f"Google Colab con {ent['vcpu']} vCPU y {ent['ram_gb']} GB de RAM; Python {ent['python']}, "
                      f"pandas {ent['pandas']} (McKinney, 2010; The pandas development team, s. f.), Polars "
                      f"{ent['polars']} (Polars Developers, s. f.) y PySpark {ent['pyspark']} (Apache Software "
                      "Foundation, 2025). Spark se ejecutó en modo local, en un solo nodo, no en un clúster."),
        ('Medición. ', f"La función measure_performance() ejecuta cada repetición en un proceso nuevo, con un "
         f"calentamiento descartado y {k['b']['repeticiones']} repeticiones medidas por herramienta y modo, en orden "
         "rotado y con un reloj monotónico de alta resolución. La memoria se muestrea cada 20 ms con psutil (Rodola, "
         "s. f.): porcentaje de memoria del sistema y RSS del proceso con sus procesos hijos, incluida la JVM."),
        ('Validez. ', 'Las salidas de las tres herramientas son equivalentes en todos los modos: mismas filas, mismo '
                      'esquema lógico, mismos conteos y misma huella SHA-256 del contenido.' if k['b']['equivalentes']
         else 'La comprobación de equivalencia detectó diferencias; véase la hoja Equivalencia del reporte.'),
    ])

    d.h1('Resultados del benchmark')
    filas = []
    for h in ('pandas', 'polars', 'spark'):
        filas.append([NOMBRE[h], f"{f(e2e.loc[h, 'seg'], 3)} ± {f(e2e.loc[h, 'seg_de'], 3)}",
                      f(pop.loc[h, 'seg'], 3), f(cal.loc[h, 'seg'], 3) if cal is not None else '—',
                      f(e2e.loc[h, 'rss'], 0), f(e2e.loc[h, 'sis'], 1), str(int(matriz['loc'][h]))])
    d.tabla(['Herramienta', 'Extremo a extremo (s, M ± DE)', 'Por operación (s)', 'Proceso caliente (s)',
             'RSS pico (MB)', 'Memoria del sistema (%)', 'Líneas de código'],
            filas, [2.4, 3.3, 2.2, 2.2, 2.0, 2.4, 1.8],
            titulo='Tiempo total de la carga a la exportación, memoria y líneas de código por herramienta',
            nota=f"M = media; DE = desviación estándar; {k['b']['repeticiones']} repeticiones por herramienta. El proceso "
                 "caliente (modo complementario) mide una segunda ejecución dentro del mismo proceso, tras limpiar la "
                 "caché de datos. Los tiempos excluyen el arranque de la sesión de Spark.")
    d.imagen(rutas.get('g_totales_doc', rutas['g_totales']), 16, 'Tiempo total por modo y memoria pico por herramienta',
             'Las barras muestran la media y las líneas de error, la desviación estándar.')
    ops = k['b']['ops']
    d.p('Por operación (modo por_operacion), los tiempos medios fueron los siguientes: ' + '; '.join(
        f"{op}: " + ', '.join(f"{NOMBRE[h]} {f(ops.loc[op, h], 3)} s" for h in ('pandas', 'polars', 'spark'))
        for op in ('carga', 'filtrado', 'agregacion', 'join', 'exportacion')) + '.')
    if cal is not None:
        d.p(f"En régimen estable (proceso caliente), PySpark desciende de {f(e2e.loc['spark', 'seg'], 2)} s a "
            f"{f(cal.loc['spark', 'seg'], 2)} s. La diferencia corresponde a los costes de una JVM recién creada (carga "
            "de clases, compilación JIT y generación de código), que no dependen del volumen. Aun así, PySpark resulta "
            f"{f(veces(cal.loc['spark', 'seg'], cal['seg'].min()), 0)} veces más lento que la opción más rápida, porque "
            "con unas 27 000 filas el trabajo útil es mínimo frente a la planificación, el intercambio de datos entre "
            "particiones y la coordinación de tareas.")

    d.h1('Matriz de decisión ponderada')
    filas = [[c, f"{int(round(p * 100))} %"] + [str(matriz['puntajes'][c][h]) for h in ('pandas', 'polars', 'spark')]
             for c, p in matriz['pesos'].items()]
    filas.append(['Puntaje ponderado', '100 %'] + [f(matriz['ponderado'][h], 2) for h in ('pandas', 'polars', 'spark')])
    d.tabla(['Criterio', 'Peso', 'Pandas', 'Polars', 'PySpark'], filas, [5, 2, 2.6, 2.6, 2.6],
            titulo='Matriz de decisión tecnológica ponderada (escala de 1 a 5)',
            nota='Puntaje ponderado = Σ (puntaje del criterio × peso). Los pesos son los establecidos en la guía de la '
                 'actividad. El cálculo se realiza con fórmulas en reporte_comparativo_benchmark.xlsx (hoja Matriz).')
    d.vinetas([
        ('Criterios cuantitativos. ', 'Se calcula r = valor / mejor valor y se asigna: r ≤ 1.25 → 5; ≤ 2 → 4; ≤ 4 → 3; '
         '≤ 8 → 2; > 8 → 1. Rendimiento usa el tiempo total extremo a extremo y Recursos, el RSS pico (proceso y JVM). '
         'Facilidad de uso parte del cociente de líneas de código, con un ajuste cualitativo de −1 para PySpark por la '
         'configuración de la sesión y de la JVM, las trazas de error mixtas Java/Python y la evaluación perezosa.'),
        ('Criterios cualitativos. ', 'Escalabilidad: Pandas 2 (un nodo, en memoria), Polars 3 (un nodo, multihilo, '
         'procesamiento por flujos) y PySpark 5 (distribuido). Curva de aprendizaje: Pandas 5, Polars 4 y PySpark 2, '
         'por los conceptos de DAG, evaluación perezosa, particiones y caché.'),
    ])

    d.h1('Conveniencia para el volumen probado frente a la elección exigida')
    d.p(f"El resultado numérico favorece a {NOMBRE[k['ganador']]}. Con este volumen predominan los costes fijos de "
        "Spark: arranque de la JVM, planificación del DAG, serialización entre Python y la JVM, y tareas con particiones "
        "casi vacías. Pandas y Polars se ejecutan dentro del mismo proceso y no incurren en esos costes. Esta conclusión "
        "es válida para unos 3 MB en un nodo y no se extiende automáticamente a volúmenes mayores.")
    d.p("La elección de PySpark como herramienta principal, exigida por la actividad, se sostiene en atributos que un "
        "benchmark de tres meses en un solo nodo no mide: el mismo código se ejecuta en un clúster repartiendo datos y "
        "cómputo entre nodos, tolera fallos de tareas, no está limitado por la RAM de una máquina porque derrama a disco "
        "e integra lectura y escritura particionada en almacenamiento de objetos (Zaharia et al., 2016). La matriz "
        "pondera la escalabilidad con un 20 %, pero ese atributo solo se manifiesta con volúmenes y despliegues que esta "
        "prueba de concepto (PoC) no alcanza.")

    d.h1('Recomendaciones por volumen')
    esc = R.get('escala')
    if esc:
        a = esc['ajustes']['pandas']
        d.p(f"En el PoC con datos replicados, el RSS de Pandas creció unos {f(a['rss_por_millon_mb'], 0)} MB por cada "
            "millón de registros (ajuste lineal sobre tres puntos medidos). La cifra orienta el orden de magnitud y no "
            "constituye un umbral garantizado.")
    d.tabla(['Escenario de datos', 'Herramienta recomendada', 'Justificación'], [
        ['Exploración y reportes de pocos meses (los datos y sus copias intermedias caben holgadamente en la RAM de un '
         'nodo)', 'Polars (o Pandas)', 'Menor latencia y menor memoria medidas; sin JVM.'],
        ['Histórico de uno a dos años en un nodo (Colab, ~12 GB de RAM)', 'PySpark local o Polars',
         'Ambos procesan el volumen replicado medido; PySpark conserva el código listo para el clúster.'],
        ['Varios años, varias fuentes, procesos programados o datos que no caben en un nodo',
         'PySpark en clúster (p. ej., Databricks)', 'Paralelismo entre nodos, tolerancia a fallos y derrame a disco.'],
    ], [6.5, 4, 6], titulo='Herramienta recomendada según el volumen de datos')

    d.h1('Herramientas complementarias')
    d.vinetas([
        ('Polars y Pandas: ', 'exploración rápida, validaciones puntuales y preparación de reportes sobre los Parquet agregados.'),
        ('PyArrow y Parquet: ', 'formato columnar común a las tres herramientas; las salidas se leen igual desde cualquiera.'),
        ('Google Drive y Google Colab: ', 'almacenamiento y cómputo del PoC; en producción, almacenamiento de objetos y un clúster.'),
        ('Matplotlib, openpyxl y python-docx: ', 'gráficos e informes regenerables a partir de las mediciones.'),
    ])

    d.h1('Plan de implementación por fases')
    d.tabla(['Fase', 'Alcance', 'Criterio de salida'], [
        ['0. PoC (realizada)', 'Pipeline PySpark en Colab: ingesta desde Drive, validaciones, Parquet particionado y '
         'métricas.', f"Tasa de validaciones del {f(ps['tasa_validaciones_pct'], 1)} % y salidas equivalentes a Pandas."],
        ['1. Consolidación', 'Carga incremental mensual, catálogo oficial de códigos (sustituye al derivado) y '
         'parametrización por entorno.', 'Mismo resultado al añadir un mes sin reprocesar el histórico.'],
        ['2. Escalado', 'Mismo código en Databricks Community u otro clúster Spark; almacenamiento de objetos; ajuste '
         'de particiones y memoria.', 'Tiempo y costo medidos con al menos un año de datos reales.'],
        ['3. Operación', 'Orquestación programada, alertas por controles fallidos y tablero de indicadores.',
         'Ejecuciones automáticas con registro de validación e indicadores.'],
    ], [3, 8.5, 5], titulo='Plan de implementación y migración por fases')

    d.h1('Limitaciones')
    d.vinetas([
        'El volumen es pequeño: predominan los costes fijos y la extrapolación lineal a volúmenes grandes no garantiza '
        'el rendimiento.',
        'Colab ofrece un único nodo de 2 vCPU, por lo que no se midió Spark distribuido.',
        'El porcentaje de memoria del sistema incluye otros procesos de Colab; el RSS contabiliza páginas compartidas, '
        'y el muestreo de 20 ms puede omitir picos más breves.',
        'Tras el calentamiento, la caché de páginas del sistema operativo quedó igualmente activa para las tres herramientas.',
        'El catálogo de tipos es derivado de los datos y no oficial; la escalabilidad cualitativa se basa en la '
        'arquitectura de cada herramienta, no en mediciones multinodo.',
    ])
    d.referencias([refs[c] for c in ('guia', 'datos', 'spark', 'parquet', 'colab', 'mckinney', 'pandas', 'polars',
                                      'psutil', 'zaharia')], tam=9.5)
    return d.guardar(ruta)


# ---------------------------------------------------------------- Arquitectura (5–8 páginas)
def doc_arquitectura(R, matriz, rutas, config_spark, ruta, datos):
    k = _datos_clave(R, matriz)
    poc, ps, pp, ent = k['poc'], k['ps'], k['pp'], k['ent']
    datos = {**datos, 'fecha': fecha_es(ent['fecha'])}
    refs = referencias_apa(datos, ent)
    u = poc['ultima_spark']
    var = poc['variante'].iloc[0] if len(poc['variante']) else None
    d = Doc(interlineado=1.0)
    d.bloque_titulo('Arquitectura técnica del sistema de análisis de emergencias de la Línea 123 de Bogotá: '
                    'prueba de concepto en PySpark', datos)

    d.h1('Introducción')
    d.p("Este documento describe la arquitectura implementada y validada en la prueba de concepto (PoC, por sus siglas "
        "en inglés) del pipeline de procesamiento de las llamadas a la Línea 123 de Bogotá. La arquitectura traduce "
        "los resultados del análisis comparativo de la primera actividad en decisiones concretas: PySpark como motor "
        f"principal, exigido por la actividad {_cita_guia(datos, ent)}; Parquet como formato de almacenamiento "
        "(Apache Software Foundation, s. f.), y validaciones de calidad explícitas. Todo el flujo se ejecuta en Google "
        "Colab con los datos en Google Drive (Google, s. f.); las cifras provienen de tres ejecuciones reales del PoC.")
    d.h1('Objetivos')
    d.vinetas([
        'Ingerir los tres archivos CSV mensuales desde Google Drive con un esquema explícito y conservar su procedencia.',
        'Limpiar, tipar y estandarizar los datos sin imputar valores: solo se eliminan las copias duplicadas exactas, '
        'que quedan registradas aparte, y las demás anomalías se conservan con una observación de calidad.',
        'Enriquecer los registros con el catálogo de tipos mediante una unión por difusión (broadcast join) y generar '
        'agregados por periodo, localidad y tipo.',
        'Exportar a Parquet particionado y verificar las salidas mediante conteos, esquema y agregados.',
        'Medir el tiempo, la memoria, el throughput y la tasa de validaciones, y compararlos con Pandas en igualdad de alcance.',
    ])

    d.h1('Arquitectura propuesta')
    d.imagen(rutas['diagrama'], 15, 'Arquitectura implementada: ingesta, procesamiento, almacenamiento y salidas',
             'Elaboración propia. Fuente editable en diagrama_arquitectura.drawio (draw.io).')
    d.p("La arquitectura se organiza en cuatro capas. En la ingesta, los CSV mensuales residen en Google Drive con su "
        "huella SHA-256 verificada y la máquina virtual de Colab los lee mediante el montaje de Drive. El procesamiento "
        "consiste en un pipeline de PySpark ejecutado en modo local (Apache Software Foundation, 2025). El almacenamiento "
        "escribe en Drive el detalle en Parquet particionado por periodo, los agregados, las copias duplicadas "
        "eliminadas, las mediciones y los registros de validación. Las salidas comprenden los indicadores, los informes y el consumo analítico de los "
        "Parquet.")

    d.h1('Componentes técnicos')
    d.tabla(['Componente', 'Tecnología', 'Función'], [
        ['Fuente de datos', 'CSV mensuales de la Línea 123 (UTF-8 con BOM, «;»)',
         f"{f(R['inventario']['total_registros'], 0)} registros de julio a septiembre de 2026"],
        ['Almacenamiento', 'Google Drive (MyDrive/BigData_U2)', 'Carpeta única: datos/, resultados/ y una carpeta por actividad'],
        ['Cómputo', f"Google Colab, {ent['vcpu']} vCPU, {ent['ram_gb']} GB de RAM", 'Máquina virtual de un solo nodo'],
        ['Motor', f"PySpark {ent['pyspark']} sobre {java_corto(ent['java'])}", 'Driver y ejecutores en una misma JVM (modo local)'],
        ['Código', 'Dos notebooks de Colab', 'Procesamiento (pipeline, validaciones y mediciones) e informes'],
        ['Configuración', 'config.yaml y catálogos CSV', 'Esquema, periodos, reglas y parámetros de Spark'],
        ['Catálogos', 'Localidades de Bogotá (oficial) y tipos de incidente (derivado, no oficial)',
         'Estandarización y enriquecimiento'],
        ['Formato', 'Apache Parquet (snappy)', 'Almacenamiento columnar, tipado y particionado'],
        ['Monitoreo', 'psutil y reloj perf_counter_ns', 'Tiempo, porcentaje de RAM del sistema y RSS del proceso y la JVM'],
    ], [3.4, 6.6, 6], titulo='Componentes técnicos de la arquitectura')

    d.h1('Flujo de datos')
    d.vinetas([
        ('Prevalidación: ', 'existencia de los tres archivos, columnas requeridas y filas físicas por archivo.'),
        ('Lectura: ', 'esquema explícito con diez columnas de texto y _corrupt_record, en modo PERMISSIVE.'),
        ('Procedencia: ', 'archivo de origen (metadato del archivo) y periodo AAAA-MM según la configuración.'),
        ('Nulos: ', '«N/A» y los vacíos se convierten en nulos, sin imputación; una RECEPCION vacía es un valor legítimo.'),
        ('Tipos: ', 'try_to_timestamp para las fechas y try_cast para los enteros, porque Spark 4 activa el modo ANSI; '
         'la edad se expresa en años según la unidad.'),
        ('Estandarización: ', 'localidad sin tildes y en mayúsculas, contrastada con la tabla oficial código-nombre; '
         'el código de tipo es el texto previo a « - ».'),
        ('Observaciones de calidad: ', 'registro corrupto, incidente nulo, fecha inválida, localidad sin dato o fuera '
         'del catálogo, código inconsistente, tipo nulo y edad fuera del rango 0–120; el registro se conserva y se marca.'),
        ('Duplicados: ', 'se eliminan solo las copias con el 100 % de los atributos iguales y se escriben aparte; los '
         'incidentes con varios despachos se conservan y se informan.'),
        ('Enriquecimiento y agregados: ', 'unión por difusión con el catálogo (23 filas), verificando que no se '
         'multipliquen filas; agregados por periodo, localidad, tipo, prioridad y día.'),
        ('Escritura y verificación: ', 'Parquet particionado por periodo, relectura con un lector independiente '
         '(PyArrow), evaluación de los controles V01–V20 y registro .txt.'),
    ])

    d.h1('Decisiones de diseño')
    d.tabla(['Decisión', 'Alternativa descartada', 'Justificación'], [
        ['Esquema explícito y tipado posterior', 'Inferencia de esquema', 'EDAD contiene «N/A»; inferir tipos fallaría '
         'o degradaría la columna a texto sin control.'],
        ['try_cast y try_to_timestamp', 'Conversión directa', 'Con ANSI activo, una conversión inválida aborta el '
         'trabajo; con try_* se obtiene un nulo y el registro se conserva con una observación de calidad.'],
        ['Conservar y marcar las anomalías', 'Filtrar y descartar', 'No se pierde información; la columna '
         'observacion_calidad permite excluirlas en cada análisis.'],
        ['Eliminar solo duplicados exactos y escribirlos aparte', 'Eliminar por número de incidente', 'Una fila idéntica '
         'en los diez atributos no aporta información; los despachos múltiples difieren en algún atributo y se conservan. '
         'Se cumple que leídos = conservados + eliminados.'],
        ['Unión por difusión del catálogo', 'Unión con ordenamiento y mezcla', 'El catálogo tiene 23 filas; la difusión '
         'evita el intercambio de datos del lado grande.'],
        ['Partición por periodo', 'Partición por periodo y localidad', 'Con unas 9 000 filas por mes, la partición '
         'doble genera archivos diminutos (véase la Tabla 4).'],
        ['Formato Parquet', 'CSV', 'Columnar, comprimido y tipado; permite poda de particiones.'],
        ['Validar sobre la relectura del Parquet', 'Validar en memoria', 'Comprueba lo que efectivamente quedó escrito.'],
        ['Memoria del driver: mín(4g, 50 % de la RAM)', '70 % de la RAM disponible (ejemplo del curso)', 'La guía fija '
         '4g; el margen restante queda para los procesos de Python y para las mediciones con Pandas y Polars.'],
        ['Serializador por defecto', 'KryoSerializer (ejemplo del curso)', 'Kryo acelera RDD y clausuras; la API de '
         'DataFrames serializa con los codificadores de Tungsten, por lo que no aporta en este pipeline.'],
    ], [4.2, 3.4, 8.4], titulo='Decisiones de diseño y su justificación')

    d.h1('Configuración de PySpark')
    explicacion = {
        'spark.master': 'Modo local con un hilo por vCPU (un nodo; no es un clúster)',
        'spark.driver.memory': 'Mínimo entre 4g (guía) y el 50 % de la RAM detectada',
        'spark.driver.maxResultSize': 'Límite para collect() (guía)',
        'spark.sql.adaptive.enabled': 'Ejecución adaptativa (AQE): ajusta el plan con estadísticas reales',
        'spark.sql.adaptive.coalescePartitions.enabled': 'Une particiones pequeñas tras el intercambio',
        'spark.sql.adaptive.skewJoin.enabled': 'Divide particiones sesgadas en las uniones',
        'spark.sql.adaptive.advisoryPartitionSizeInBytes': 'Tamaño objetivo de partición (guía)',
        'spark.sql.execution.arrow.pyspark.enabled': 'Arrow para el intercambio con pandas',
        'spark.sql.shuffle.partitions': '2 × vCPU, en lugar de 200 particiones casi vacías',
        'spark.sql.session.timeZone': 'Zona horaria de los datos',
        'spark.ui.enabled': 'Desactivada para no medir el servidor de la interfaz',
    }
    d.tabla(['Parámetro', 'Valor', 'Motivo'], [[k2, v, explicacion.get(k2, '')] for k2, v in config_spark.items()
                                              if k2 not in ('spark.app.name', 'spark.ui.showConsoleProgress')],
            [6.3, 2.5, 7.2], tam=8.5, titulo='Configuración de PySpark ajustada a los recursos de Colab')
    d.p("En Colab, el driver y los ejecutores comparten una sola JVM en una máquina virtual. En un clúster, "
        "spark.master apuntaría al gestor de recursos (YARN, Kubernetes o Databricks); spark.executor.instances, "
        "spark.executor.cores y spark.executor.memory se dimensionarían según el volumen, y "
        "spark.sql.shuffle.partitions se ajustaría al total de núcleos, sin cambios en el código del pipeline.")

    d.h1('Estrategia de particionamiento')
    filas = [['Periodo (implementada)', str(int(u['parquet_archivos'])), f(u['parquet_bytes'] / 1024, 0),
              f(u['parquet_bytes'] / u['parquet_archivos'] / 1024, 0)]]
    if var is not None:
        filas.append(['Periodo y localidad (variante medida)', str(int(var['parquet_archivos'])),
                      f(var['parquet_bytes'] / 1024, 0), f(var['parquet_bytes'] / var['parquet_archivos'] / 1024, 0)])
    d.tabla(['Estrategia', 'Archivos Parquet', 'KB totales', 'KB por archivo'], filas, [6, 3.2, 3.2, 3.2],
            titulo='Comparación medida de las estrategias de particionamiento',
            nota='La variante por periodo y localidad corresponde a una ejecución de referencia y no entra en los promedios.')
    d.p("El detalle se escribe con repartition(periodo), sortWithinPartitions(localidad_norm, fecha_inicio) y "
        "partitionBy(periodo): un archivo por mes, ordenado por localidad. Las consultas por mes descartan particiones "
        "completas y las consultas por localidad aprovechan las estadísticas mínimas y máximas de los grupos de filas. "
        "La variante por periodo y localidad multiplica los archivos y reduce su tamaño medio, lo que origina el "
        "problema de los archivos pequeños sin un beneficio real de poda con este volumen. Esa partición se justificaría "
        "cuando cada combinación de mes y localidad alcance decenas de megabytes.")

    optim = R.get('optim')
    if optim:
        d.h1('Evidencia de optimización')
        pl = optim['planes']
        si_no = {True: 'sí', False: 'no'}
        d.p("Los planes físicos obtenidos con explain() sobre el Parquet exportado confirman tres optimizaciones: "
            f"poda de particiones por periodo ({si_no[pl['poda_particiones']]}), predicado empujado al lector Parquet "
            f"sobre la localidad ({si_no[pl['predicado_empujado']]}) y unión por difusión con el catálogo "
            f"({si_no[pl['broadcast_join']]}). El texto completo de los planes se conserva en "
            "resultados/mediciones/planes_ejecucion.txt.")
        filas = [[x['formato'], f(x['mb'], 3), f(x['tamano_vs_csv'], 2), f(x['lectura_agregacion_s'], 2)]
                 for x in optim['formatos']['formatos']]
        d.tabla(['Formato', 'Tamaño (MB)', 'Tamaño relativo a CSV', 'Lectura y agregación (s)'], filas, [4, 3.5, 4, 4.5],
                titulo='Comparación de formatos de almacenamiento con el detalle del PoC',
                nota=f"Mismo contenido ({f(optim['formatos']['registros'], 0)} registros) escrito en un archivo por "
                     f"formato; tiempo = mediana de {optim['formatos']['repeticiones']} lecturas con agregación por tipo "
                     "en Spark. Diagnóstico adicional: perfil de nulos por columna y valores atípicos de la edad por "
                     f"rango intercuartílico ({optim['atipicos_edad']['atipicos']} registros fuera de "
                     f"[{f(optim['atipicos_edad']['limite_inferior'], 1)}, {f(optim['atipicos_edad']['limite_superior'], 1)}] "
                     "años), solo informativo.")

    d.h1('Recursos medidos y estimación')
    d.tabla(['Métrica (media de tres ejecuciones)', 'PySpark', 'Pandas (mismo alcance)'], [
        ['Tiempo total de la carga a la exportación (s)', f(ps['tiempo_total_s'], 2), f(pp['tiempo_total_s'], 2)],
        ['Arranque de la sesión de Spark (s, aparte)', f(ps['tiempo_inicio_sesion_s'], 2), '—'],
        ['Memoria del sistema, pico (%)', f(ps['mem_sis_pico_pct'], 1), f(pp['mem_sis_pico_pct'], 1)],
        ['RSS pico del proceso y la JVM (MB)', f(ps['rss_pico_mb'], 0), f(pp['rss_pico_mb'], 0)],
        ['Throughput (registros por minuto)', f(ps['throughput_reg_min'], 0), f(pp['throughput_reg_min'], 0)],
        ['Tasa de éxito de validaciones (%)', f(ps['tasa_validaciones_pct'], 1), f(pp['tasa_validaciones_pct'], 1)],
        ['Registros conservados (%)', f(ps['registros_validos_pct'], 3), f(pp['registros_validos_pct'], 3)],
    ], [7.5, 4, 4.5], titulo='Recursos medidos en la prueba de concepto')
    d.imagen(rutas['g_poc'], 14, 'Prueba de concepto en PySpark frente a Pandas con el mismo alcance',
             'Media de tres ejecuciones; las líneas de error representan la desviación estándar.')
    d.p(f"PySpark resultó {f(veces(ps['tiempo_total_s'], pp['tiempo_total_s']), 0)} veces más lento que Pandas y "
        f"empleó {f(veces(ps['rss_pico_mb'], pp['rss_pico_mb']), 1)} veces más RSS. Con este volumen no hubo mejora "
        "frente a Pandas, debido a los costes fijos de la JVM y de la coordinación de tareas. Para el PoC bastan 2 vCPU "
        "y unos 4 GB de memoria para el driver; la proyección a uno y dos años se presenta en el análisis de escalabilidad.")

    d.h1('Monitoreo e indicadores clave')
    d.tabla(['Indicador', 'Definición', 'Valor medido (PySpark)'], [
        ['Tiempo total (s)', 'Reloj monotónico de la carga a la exportación, sin el arranque', f(ps['tiempo_total_s'], 2)],
        ['Memoria pico (%)', 'Máximo de psutil.virtual_memory().percent, muestreado cada 20 ms', f(ps['mem_sis_pico_pct'], 1)],
        ['RSS pico (MB)', 'Máximo del RSS del proceso y sus hijos (JVM)', f(ps['rss_pico_mb'], 0)],
        ['Throughput', 'Registros leídos / (tiempo total / 60)', f(ps['throughput_reg_min'], 0) + ' registros/min'],
        ['Tasa de validaciones (%)', 'Controles aprobados / evaluados (bloqueantes y advertencias)', f(ps['tasa_validaciones_pct'], 1)],
        ['Registros conservados (%)', 'Conservados / leídos', f(ps['registros_validos_pct'], 3)],
        ['Duplicados exactos', 'Copias idénticas eliminadas; registradas en duplicados_eliminados/',
         str(int(u['duplicados_exactos']))],
        ['Observaciones de calidad', 'Registros conservados con anomalía, por motivo',
         ', '.join(f'{k}: {v}' for k, v in (u.get('observaciones_por_motivo') or {}).items()) or 'ninguna'],
    ], [3.6, 8.4, 4], titulo='Indicadores de monitoreo del pipeline')
    d.p("Cada ejecución agrega sus métricas a resultados/mediciones/poc.jsonl y su bloque de controles al registro "
        "validacion_poc_spark.txt. En operación, un control bloqueante fallido debería detener la publicación del mes y "
        "generar una alerta.")

    d.h1('Riesgos y mitigaciones')
    d.tabla(['Riesgo', 'Impacto', 'Mitigación'], [
        ['Cambio de esquema o de formato en un CSV nuevo', 'Fallo de lectura o columnas vacías',
         'Control V02 y columna _corrupt_record; el esquema explícito evita inferencias silenciosas.'],
        ['Catálogo de códigos no oficial', 'Descripciones incompletas', 'Sustituirlo por el catálogo oficial con el '
         'mismo formato; el control V08 cuenta los códigos no reconocidos.'],
        ['Códigos de localidad inconsistentes', 'Asignación a una localidad equivocada', 'Observación de calidad y control V10.'],
        ['Archivos pequeños por sobreparticionar', 'Lecturas lentas', 'Partición solo por periodo; revisión con el volumen real.'],
        ['Límites de Colab (un nodo, sesión de 12 h)', 'Sesiones interrumpidas', 'Resultados persistidos en Drive; '
         'migración a un clúster en la fase 2.'],
        ['Costes fijos de Spark con poco volumen', 'Latencia alta en lotes pequeños', 'Polars o Pandas para lotes '
         'pequeños y Spark para el histórico y el clúster.'],
        ['Autorización de Drive caducada', 'Pipeline sin datos', 'Control V01 con error explícito y copia verificada por SHA-256.'],
    ], [5, 3.8, 7.2], titulo='Riesgos identificados y medidas de mitigación')

    d.h1('Limitaciones y siguientes pasos')
    d.p("Lo implementado es un PoC de un solo nodo, por lo que no se midió el comportamiento distribuido. Las cifras "
        f"corresponden a {f(R['inventario']['total_registros'], 0)} registros y su relación con volúmenes mayores es una "
        "estimación. Como siguientes pasos se proponen el catálogo oficial, la carga incremental mensual, la ejecución en "
        "Databricks o en un clúster con al menos un año de datos reales, y la orquestación con alertas basadas en los "
        "controles V01–V16 (Zaharia et al., 2016).")
    d.referencias([refs[c] for c in ('guia', 'datos', 'spark', 'parquet', 'colab', 'psutil', 'zaharia')], tam=9.5)
    return d.guardar(ruta)


# ---------------------------------------------------------------- Escalabilidad (1 página)
def doc_escalabilidad(R, ruta, datos):
    esc, ent = R['escala'], R['entorno']
    datos = {**datos, 'fecha': fecha_es(ent['fecha'])}
    refs = referencias_apa(datos, ent)
    d = Doc(tamano=8.5, margen_cm=1.6, interlineado=1.0)
    d.bloque_titulo('Análisis de escalabilidad del pipeline: proyección a uno y dos años de datos', datos, compacto=True)
    d.p('Se midió la prueba de concepto (PoC) con los datos reales de tres meses y con los mismos CSV replicados ×4 y ×8 '
        '(datos sintéticos equivalentes en filas a uno y dos años). Con esos puntos se ajustó, para cada herramienta, '
        'el modelo tiempo = fijo + variable × registros, y de igual forma la memoria residente del proceso (RSS). Las '
        'proyecciones fuera de los puntos medidos son teóricas.', 'Método. ')
    filas = []
    for _, r in esc['puntos'].iterrows():
        filas.append([NOMBRE[r['herramienta']], f"{3 * r['factor']} meses (×{r['factor']})", f(r['leidos'], 0),
                      f(r['tiempo_total_s'], 2), f(r['rss_pico_mb'], 0), f(r['mem_sis_pico_pct'], 1), r['origen']])
    d.tabla(['Herramienta', 'Volumen', 'Registros', 'Tiempo (s)', 'RSS pico (MB)', 'Memoria del sistema (%)', 'Origen'],
            filas, [2.2, 2.6, 2.2, 2.0, 2.3, 2.9, 3.4], tam=7.5, titulo='Puntos medidos de tiempo y memoria según el volumen')
    aj = esc['ajustes']
    d.p('; '.join(f"{NOMBRE[h]}: tiempo ≈ {f(a['t_fijo_s'], 1)} s + {f(a['t_por_millon_s'], 1)} s por millón de "
                  f"registros; RSS ≈ {f(a['rss_fijo_mb'], 0)} MB + {f(a['rss_por_millon_mb'], 0)} MB por millón"
                  for h, a in aj.items()) + '.', 'Ajuste lineal sobre los puntos medidos. ')
    n1, n2, n10 = REGISTROS_MES * 12, REGISTROS_MES * 24, REGISTROS_MES * 120
    heap_mb, proy, acotado = 4 * 1024, [], False
    for etiqueta, n in (('1 año', n1), ('2 años', n2), ('10 años (teórico)', n10)):
        fila = [etiqueta, f(n, 0)]
        for h in ('spark', 'pandas'):
            t, m = proyectar(aj[h], n)
            if h == 'spark' and m > heap_mb:
                acotado = True
                fila += [f(t, 1), f'≥ {f(heap_mb, 0)}*']
            else:
                fila += [f(t, 1), f(m, 0)]
        proy.append(fila)
    d.tabla(['Horizonte', 'Registros (~9 145 por mes)', 'PySpark (s)', 'PySpark (MB)', 'Pandas (s)', 'Pandas (MB)'],
            proy, [3, 4, 2.3, 2.3, 2.3, 2.3], tam=7.5, titulo='Proyección teórica de tiempo y memoria',
            nota=('* La memoria de Spark no crece sin límite: la JVM se acota al heap configurado (spark.driver.memory '
                  '= 4 GB) más el overhead y, por encima, derrama a disco; la proyección lineal del RSS de Spark no aplica '
                  'en ese tramo.') if acotado else None)
    ram_mb = ent['ram_gb'] * 1024
    quiebre = {h: (0.5 * ram_mb - a['rss_fijo_mb']) / a['rss_por_millon_mb'] * 1e6 if a['rss_por_millon_mb'] > 0
               else float('inf') for h, a in aj.items()}
    pts = esc['puntos']
    d.p(f"Uno a dos años de datos ({f(n1 / 1e3, 0)}–{f(n2 / 1e3, 0)} mil registros; unos {f(n1 * 113 / 2**20, 0)}–"
        f"{f(n2 * 113 / 2**20, 0)} MB en CSV) caben en un nodo de Colab con ambas herramientas: los puntos ×4 y ×8 se "
        f"midieron y no superaron el {f(pts['mem_sis_pico_pct'].max(), 0)} % de la RAM del sistema. Con este volumen, "
        "el coste fijo de Spark (JVM y planificación) sigue superando al variable, por lo que Pandas continúa siendo "
        "más rápido en un solo nodo; Polars no se midió en este experimento.", 'Interpretación. ')
    d.p(f"Si el ajuste lineal se mantuviera, supuesto que es fuerte, Pandas alcanzaría el 50 % de la RAM de la máquina "
        f"virtual (unos {f(0.5 * ram_mb / 1024, 1)} GB, con margen para copias intermedias) hacia los "
        f"{f(quiebre['pandas'] / 1e6, 0)} millones de registros (≈ {f(quiebre['pandas'] / REGISTROS_MES / 12, 0)} años "
        "al ritmo actual), y no puede superar la RAM de la máquina. Spark, en cambio, derrama a disco y en un clúster "
        "incorpora nodos (Zaharia et al., 2016); su límite práctico en Colab es la CPU (2 vCPU) y la duración de la "
        "sesión (12 h). Estos umbrales indican órdenes de magnitud, no garantías.", 'Puntos de quiebre teóricos. ')
    d.p('Hasta dos años basta un nodo de Colab (2 vCPU, 12 GB y driver de Spark de 4 GB). Para varios años, varias '
        'fuentes o ejecución diaria se recomienda un clúster Spark (p. ej., Databricks) con dos a cuatro ejecutores y '
        'particionado mensual, dimensionado tras medir con datos reales, porque los replicados no reproducen la variedad '
        'ni la distribución real de los valores.', 'Recursos recomendados. ')
    d.referencias([refs['spark'], refs['zaharia']], tam=7.5)
    return d.guardar(ruta)

**Documento de sustentación (`sustentacion.py`).** Explica el proyecto completo a una persona sin
conocimientos técnicos.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/sustentacion.py
"""Documento de sustentación: explica el proyecto completo a una persona sin conocimientos técnicos.

Toda cifra se toma de las mediciones (resumen.cargar_todo) y de la matriz recalculada, igual que los
demás informes. Formato APA 7 (bloque de título, encabezados sin numerar, tablas y figuras numeradas,
referencias), con un lenguaje sencillo, analogías y un glosario.
"""
from .documentos import (NOMBRE, Doc, _cita_guia, _datos_clave, f, fecha_es, java_corto, referencias_apa,
                         veces)


def doc_sustentacion(R, matriz, rutas, ruta, datos):
    k = _datos_clave(R, matriz)
    e2e, cal, ent, ps, pp, poc = k['e2e'], k['cal'], k['ent'], k['ps'], k['pp'], k['poc']
    inv, b = R['inventario'], k['b']
    datos = {**datos, 'fecha': fecha_es(ent['fecha'])}
    refs = referencias_apa(datos, ent)
    total = inv['total_registros']
    rapida, ganador = k['rapida'], k['ganador']
    orden = sorted(matriz['ponderado'], key=matriz['ponderado'].get, reverse=True)
    u = poc['ultima_spark']
    var = poc['variante'].iloc[0] if len(poc['variante']) else None
    esc, optim = R.get('escala'), R.get('optim')

    d = Doc(tamano=11, interlineado=1.15)
    d.bloque_titulo('Sustentación del proyecto: procesamiento de las llamadas a la Línea 123 de Bogotá con '
                    'Pandas, Polars y PySpark explicado paso a paso', datos)

    # ------------------------------------------------------------------ Introducción
    d.h1('Propósito de este documento')
    d.p("Este documento explica, en lenguaje sencillo, todo lo que se hizo en el proyecto de la Unidad 2 del curso "
        f"de Big Data {_cita_guia(datos, ent)}. Está pensado para una persona que no conoce de programación ni de "
        "análisis de datos. Cada término técnico se explica la primera vez que aparece; al final se incluyen un "
        "glosario y las preguntas que suelen hacerse en una sustentación, con sus respuestas.")
    d.p("La idea general cabe en una frase: se tomaron tres meses de registros de llamadas de emergencia de Bogotá, "
        "se compararon tres herramientas para procesarlos y se construyó, con la herramienta que exige la actividad "
        "(PySpark), un proceso automático que limpia los datos, comprueba su calidad y los guarda en un formato "
        "eficiente. Todo se ejecutó en Google Colab, un servicio gratuito de Google que permite programar desde el "
        "navegador sin instalar nada en el computador (Google, s. f.).")

    # ------------------------------------------------------------------ Conceptos
    d.h1('Conceptos básicos para entender el proyecto')
    d.vinetas([
        ('Datos y archivo CSV. ', 'Los datos son registros, como las filas de una hoja de cálculo. Un archivo CSV es '
         'una tabla guardada como texto, con las columnas separadas por un símbolo (aquí, el punto y coma).'),
        ('Big Data. ', 'Se habla de Big Data cuando el volumen de datos es tan grande que un solo computador no '
         'alcanza a procesarlo con comodidad y hay que repartir el trabajo.'),
        ('Procesamiento distribuido. ', 'Es repartir un trabajo grande entre varios computadores que trabajan a la '
         'vez, como un equipo de personas que se divide una montaña de papeles.'),
        ('Pandas, Polars y PySpark. ', 'Son tres herramientas del lenguaje Python para trabajar con tablas de datos. '
         'Pandas es la más conocida y trabaja en un solo computador. Polars es más moderna y aprovecha mejor los '
         'procesadores del computador. PySpark es la versión para Python de Apache Spark, una herramienta pensada '
         'para repartir el trabajo entre muchos computadores (Zaharia et al., 2016).'),
        ('Google Colab y Google Drive. ', 'Colab presta temporalmente un computador en la nube para ejecutar '
         'programas; Drive es el almacenamiento en la nube donde quedaron los datos, el código y los resultados.'),
        ('Notebook. ', 'Es un documento que mezcla explicaciones y código que se puede ejecutar por partes (celdas). '
         'El proyecto tiene dos: uno de procesamiento (comparación de herramientas y prueba de concepto) y uno de '
         'informes.'),
        ('Parquet. ', 'Es un formato de archivo que guarda los datos por columnas, comprimidos y con su tipo '
         '(número, fecha, texto). Es como una biblioteca ordenada por estantes: para buscar algo no hay que '
         'revisar todos los libros (Apache Software Foundation, s. f.).'),
        ('Memoria RAM. ', 'Es la memoria de trabajo del computador. Si un programa necesita más de la disponible, '
         'se vuelve lento o falla.'),
    ])

    # ------------------------------------------------------------------ Datos
    d.h1('Los datos utilizados')
    d.p(f"Se usaron tres archivos con las llamadas a la Línea 123 de julio, agosto y septiembre de 2026, con "
        f"{f(total, 0)} registros en total. Cada registro describe un despacho de emergencia: el número del "
        "incidente, la fecha y hora, la localidad de Bogotá, la edad, la unidad de la edad (años, meses, días u "
        "horas), el género, el tipo de incidente, la prioridad y la hora de recepción.")
    d.p('Antes de procesar los datos se revisaron con cuidado y se encontraron varias situaciones que había que '
        'resolver sin inventar información. La única eliminación permitida fue la de las filas repetidas exactamente:')
    d.tabla(['Situación encontrada', 'Qué significa', 'Cómo se resolvió'], [
        ['Valores «N/A» en edad, unidad y género', 'El dato no se registró', 'Se convierten en vacío (nulo); no se '
         'inventa un valor.'],
        ['Localidades con tilde (USAQUÉN)', 'La misma localidad podría escribirse de varias formas',
         'Se normalizan: sin tildes y en mayúsculas (USAQUEN).'],
        ['Una fila con localidad «SIN_D»', 'Localidad sin dato', 'Se conserva y se marca con una observación de '
         'calidad («localidad sin dato»), para que cada análisis decida si la usa.'],
        [f"{inv['duplicados_exactos_global']} copias repetidas exactamente", 'Filas idénticas en todos sus datos, '
         'incluida la hora al segundo', 'Se eliminan las copias (se conserva la primera) y se guardan aparte para poder '
         'revisarlas. Los incidentes con varios despachos se conservan, porque difieren en algún dato.'],
        ['Registros del día 1 del mes siguiente', 'Llamadas de la madrugada que quedaron en el archivo del mes '
         'anterior', 'Se aceptan con una tolerancia de un día y se informan.'],
        ['No existía una tabla oficial de códigos', 'No había un catálogo con el significado de cada tipo de '
         'incidente', 'Se construyó un catálogo de 23 tipos a partir de los propios datos, marcado siempre como '
         '«derivado, no oficial».'],
    ], [4.2, 4.5, 7.3], titulo='Problemas encontrados en los datos y su tratamiento')

    # ------------------------------------------------------------------ Actividad 1
    d.h1('Actividad 1: comparación de herramientas')
    d.h2('Qué se pidió')
    d.p("Comparar Pandas, Polars y PySpark haciendo exactamente las mismas cinco tareas con los mismos datos: "
        "(1) cargar y unir los tres archivos; (2) quedarse solo con las localidades de Usaquén y Chapinero; "
        "(3) contar las emergencias por tipo; (4) unir cada registro con el catálogo de tipos, y (5) guardar el "
        "resultado en Parquet. Para cada herramienta se midieron el tiempo y la memoria.")
    d.h2('Cómo se garantizó una comparación justa')
    d.vinetas([
        'Las tres herramientas recibieron los mismos archivos y aplicaron la misma lógica.',
        'Cada medición se hizo en un programa nuevo, para que ninguna herramienta reutilizara trabajo de una '
        'ejecución anterior.',
        f"Se hizo una ejecución de calentamiento, que se descarta, y luego {b['repeticiones']} repeticiones "
        "medidas por herramienta, alternando el orden.",
        'El tiempo se midió con un reloj de alta precisión y la memoria se revisó cada 20 milésimas de segundo '
        'para capturar el punto máximo.',
        'Al final se comprobó que las tres herramientas produjeron exactamente el mismo resultado: las mismas '
        '1 762 filas y la misma «huella digital» del contenido.' if b['equivalentes'] else
        'Se comprobó si las tres herramientas produjeron el mismo resultado (ver el reporte comparativo).',
    ])
    d.h2('Resultados')
    filas = [[NOMBRE[h], f(e2e.loc[h, 'seg'], 3), f(e2e.loc[h, 'rss'], 0), f(matriz['ponderado'][h], 2)]
             for h in ('pandas', 'polars', 'spark')]
    d.tabla(['Herramienta', 'Tiempo total (segundos)', 'Memoria máxima (MB)', 'Puntaje de la matriz (1 a 5)'],
            filas, [4, 4, 4, 4], titulo='Resultados principales de la comparación',
            nota=f"Tiempo de la carga a la exportación, promedio de {b['repeticiones']} repeticiones; PySpark tarda "
                 f"además {f(b['arranque_spark'], 1)} s en arrancar. MB = megabytes.")
    d.imagen(rutas.get('g_totales_doc', rutas['g_totales']), 15.5, 'Tiempo total y memoria por herramienta',
             'Barras más cortas indican mejor resultado.')
    d.p(f"La herramienta más rápida fue {NOMBRE[rapida]}. PySpark fue unas "
        f"{f(veces(e2e.loc['spark', 'seg'], e2e.loc[rapida, 'seg']), 0)} veces más lenta y usó más memoria. "
        "Esto no significa que PySpark sea una mala herramienta: está diseñada para volúmenes enormes repartidos "
        "en muchos computadores. Con solo unos 3 MB de datos, el tiempo de «encender» su maquinaria es mayor que "
        "el trabajo real. Es como usar un camión de carga para llevar un sobre a la esquina: la moto llega antes, "
        "pero el camión es el que sirve cuando hay que mover una mudanza.")
    if cal is not None:
        d.p(f"Para comprobarlo se midió también una segunda pasada dentro del mismo programa, con el motor ya "
            f"encendido: PySpark bajó de {f(e2e.loc['spark', 'seg'], 1)} s a {f(cal.loc['spark', 'seg'], 1)} s. La "
            "mayor parte del tiempo era, efectivamente, el costo de arrancar.")
    d.h2('Matriz de decisión')
    d.p("La guía pide decidir con una matriz que califica cada herramienta de 1 a 5 en cinco criterios con un "
        "peso: rendimiento (30 %), facilidad de uso (25 %), escalabilidad, es decir, capacidad de crecer (20 %), "
        "consumo de recursos (15 %) y curva de aprendizaje (10 %). Las calificaciones de rendimiento y recursos se "
        "calculan con una regla fija a partir de las mediciones; las demás se justifican por escrito. El resultado "
        "ordena las herramientas así: " + ', '.join(f"{NOMBRE[h]} ({f(matriz['ponderado'][h], 2)})" for h in orden)
        + ".")
    d.p(f"{NOMBRE[ganador]} resultó la más conveniente para este volumen pequeño. Aun así, la actividad exige "
        "PySpark como herramienta principal, y la decisión es razonable: cuando el proyecto crezca a años de datos "
        "o a varias fuentes, PySpark es la única de las tres que permite repartir el trabajo entre muchos "
        "computadores sin reescribir el código. Pandas y Polars quedan como herramientas de apoyo para análisis "
        "rápidos.")

    # ------------------------------------------------------------------ Actividad 2
    d.h1('Actividad 2: arquitectura y prueba de concepto en PySpark')
    d.h2('Qué es una arquitectura y qué es una prueba de concepto')
    d.p("La arquitectura es el «plano» del sistema: qué piezas tiene y cómo viajan los datos entre ellas. Una "
        "prueba de concepto (PoC, por sus siglas en inglés) es una versión pequeña pero funcional que demuestra que "
        "el plano funciona en la práctica.")
    d.imagen(rutas['diagrama'], 15.5, 'Plano de la arquitectura implementada',
             'De izquierda a derecha: ingesta, procesamiento, almacenamiento y salidas.')
    d.p("El plano tiene cuatro partes. En la ingesta, los archivos se guardan en Google Drive y la máquina de "
        "Colab los lee. En el procesamiento, PySpark ejecuta los pasos de limpieza y control. En el "
        "almacenamiento, los resultados quedan en Drive en formato Parquet. En las salidas se generan los "
        "indicadores, los informes y los registros de control.")
    d.h2('Los pasos del proceso automático (pipeline)')
    d.vinetas([
        ('1. Revisar los archivos: ', 'se comprueba que estén los tres archivos y que tengan las columnas esperadas.'),
        ('2. Leer con un esquema definido: ', 'se le indica a PySpark exactamente qué columnas esperar, en lugar de '
         'dejar que adivine.'),
        ('3. Anotar la procedencia: ', 'cada registro guarda de qué archivo y de qué mes viene.'),
        ('4. Tratar los vacíos: ', '«N/A» y los campos vacíos se convierten en nulos, sin inventar valores.'),
        ('5. Convertir tipos: ', 'las fechas pasan a ser fechas, las edades a números y la edad se expresa en años.'),
        ('6. Estandarizar: ', 'localidades y tipos de incidente se escriben de una sola manera.'),
        ('7. Marcar anomalías: ', 'los registros con algún problema (por ejemplo, la localidad «SIN_D») se conservan '
         'y llevan una nota que explica el problema.'),
        ('8. Quitar copias exactas: ', 'las filas idénticas en todos sus datos se eliminan y se guardan aparte.'),
        ('9. Enriquecer: ', 'cada registro recibe la descripción de su tipo de incidente desde el catálogo.'),
        ('10. Resumir: ', 'se calculan totales por mes, localidad, tipo, prioridad y día.'),
        ('11. Guardar en Parquet: ', 'el detalle se guarda en Drive, separado en una carpeta por mes.'),
        ('12. Verificar: ', 'se vuelven a leer los archivos guardados y se aplican 20 controles de calidad.'),
    ])
    d.h2('Controles de calidad')
    obs = u.get('observaciones_por_motivo') or {}
    d.p(f"Se definieron 20 controles. Por ejemplo: que los registros leídos sean iguales a los conservados más las "
        f"copias eliminadas, que ningún dato vacío importante quede sin su nota, que la unión con el catálogo no "
        f"multiplique filas y que el archivo guardado tenga exactamente los registros conservados. La tasa de éxito de "
        f"las validaciones fue del {f(ps['tasa_validaciones_pct'], 1)} %. De {f(ps['leidos'], 0)} registros se "
        f"conservaron {f(ps['aceptados'], 0)} y se eliminaron {f(ps['rechazados'], 0)} copias exactas, que quedaron "
        f"guardadas aparte. {f(sum(obs.values()), 0)} registro(s) se conservaron con una nota de calidad. Nada se "
        "borró en silencio.")
    d.h2('Cómo se organizaron los archivos guardados')
    if var is not None:
        d.p(f"Los resultados se guardaron separados por mes, lo que produjo {int(u['parquet_archivos'])} archivos. "
            f"Se probó también separar por mes y por localidad, lo que produjo {int(var['parquet_archivos'])} "
            f"archivos de unos {f(var['parquet_bytes'] / var['parquet_archivos'] / 1024, 0)} KB cada uno. Con tan "
            "pocos datos, tantos archivos diminutos hacen la lectura más lenta (el «problema de los archivos "
            "pequeños»), por lo que se eligió separar solo por mes.")
    if optim:
        pl = optim['planes']
        d.p("Además, se revisó el «plan de ejecución» de PySpark, que es la receta que el motor arma antes de "
            "trabajar. Ese plan confirma que, al pedir un solo mes, PySpark salta los demás meses sin leerlos; que "
            "el filtro por localidad se aplica al leer el archivo, y que el catálogo se envía completo a cada tarea "
            "para no mover los datos grandes" + ('.' if all(pl.values()) else ' (ver el detalle en el notebook de procesamiento).'))
        filas = [[x['formato'], f(x['mb'], 2), f(x['lectura_agregacion_s'], 2)] for x in optim['formatos']['formatos']]
        d.tabla(['Formato', 'Tamaño (MB)', 'Tiempo de lectura y conteo (s)'], filas, [5, 5, 6],
                titulo='El mismo contenido guardado en tres formatos',
                nota='Parquet ocupa mucho menos espacio que CSV y JSON porque guarda por columnas y comprimido.')
    d.h2('Métricas de la prueba de concepto')
    d.tabla(['Medida (promedio de tres ejecuciones)', 'PySpark', 'Pandas'], [
        ['Tiempo total (segundos)', f(ps['tiempo_total_s'], 2), f(pp['tiempo_total_s'], 2)],
        ['Memoria del sistema en el punto máximo (%)', f(ps['mem_sis_pico_pct'], 1), f(pp['mem_sis_pico_pct'], 1)],
        ['Registros procesados por minuto', f(ps['throughput_reg_min'], 0), f(pp['throughput_reg_min'], 0)],
        ['Tasa de éxito de validaciones (%)', f(ps['tasa_validaciones_pct'], 1), f(pp['tasa_validaciones_pct'], 1)],
    ], [8, 4, 4], titulo='Comparación de la prueba de concepto con Pandas')
    d.p(f"Como en la Actividad 1, PySpark fue más lento (unas {f(veces(ps['tiempo_total_s'], pp['tiempo_total_s']), 0)} "
        "veces) y usó más memoria que Pandas con este volumen. Los informes lo reportan tal cual: no se presenta una "
        "mejora que no existió. El valor de PySpark está en lo que permite cuando los datos crecen.")
    if esc:
        aj = esc['ajustes']['pandas']
        d.h2('Qué pasaría con uno o dos años de datos')
        d.p("Para estimarlo sin inventar, se repitieron los mismos datos 4 y 8 veces (lo que equivale en cantidad "
            "de filas a uno y dos años) y se midió de nuevo. Con esos puntos se trazó una línea de tendencia. "
            "Uno a dos años de datos todavía caben con holgura en una máquina de Colab con cualquiera de las dos "
            f"herramientas. Pandas usa unos {f(aj['rss_por_millon_mb'], 0)} MB más por cada millón de registros y, "
            "si la tendencia se mantuviera, llegaría al límite práctico de memoria de Colab con varios millones de "
            "registros. PySpark, en cambio, puede usar el disco y sumar computadores, por lo que es la opción para "
            "un crecimiento grande. Estas proyecciones son estimaciones, no garantías.")

    # ------------------------------------------------------------------ Colab
    d.h1('Cómo se trabajó: todo desde Google Colab')
    d.vinetas([
        'El proyecto son solo dos notebooks, que se suben a una única carpeta de Google Drive (BigData_U2).',
        'El primer notebook consigue los datos por sí mismo: los toma de la carpeta de Drive de la Unidad 1 o, si no '
        'están, los descarga del portal de datos abiertos, y comprueba con su «huella digital» que son los correctos.',
        'Ese mismo notebook hace todo el procesamiento, narrado paso a paso: comparación de herramientas, prueba de '
        'concepto, revisión de la optimización y pruebas automáticas. El segundo genera los informes y ordena la carpeta.',
        f"La máquina de Colab tenía {ent['vcpu']} procesadores y {ent['ram_gb']} GB de memoria, con Python "
        f"{ent['python']}, PySpark {ent['pyspark']} y {java_corto(ent['java'])}.",
        'Todo lo que pide la guía queda en esa única carpeta: los datos, una carpeta por actividad con sus informes, '
        'esta sustentación y la evidencia (mediciones, registros de control y archivos Parquet).',
    ])

    # ------------------------------------------------------------------ Entregables
    d.h1('Qué se entrega y para qué sirve cada cosa')
    d.tabla(['Entregable', 'Para qué sirve', 'Actividad'], [
        ['Notebook de procesamiento', 'Todo el trabajo, narrado y con sus resultados: comparación y prueba de concepto', '1 y 2'],
        ['Notebook de informes', 'Genera y verifica los informes y ordena la carpeta del proyecto', '1 y 2'],
        ['Reporte comparativo (Excel)', 'Mediciones, promedios, gráficos y matriz de decisión con fórmulas', '1'],
        ['Análisis de decisión (Word)', 'Explica qué herramienta conviene y por qué se mantiene PySpark', '1'],
        ['Documento de arquitectura (Word)', 'Describe el plano del sistema y sus decisiones', '2'],
        ['Diagrama de arquitectura (imagen y fuente editable)', 'Dibujo del plano del sistema', '2'],
        ['Archivos Parquet en Drive', 'Los datos limpios y verificados', '2'],
        ['Registro de validación (texto)', 'Los 20 controles de calidad y su resultado', '2'],
        ['Métricas de la prueba de concepto (Excel)', 'Las tres ejecuciones, sus promedios y la comparación', '2'],
        ['Gráfico comparativo (imagen)', 'Tiempo y memoria de PySpark frente a Pandas', '2'],
        ['Análisis de escalabilidad (Word, 1 página)', 'Qué pasaría con uno o dos años de datos', '2'],
    ], [5.5, 8, 2.5], titulo='Entregables del proyecto')

    # ------------------------------------------------------------------ Limitaciones
    d.h1('Limitaciones, explicadas con honestidad')
    d.vinetas([
        'El volumen de datos es pequeño, por lo que las ventajas de PySpark no se ven en los tiempos.',
        'Colab presta un solo computador; no se probó PySpark repartido en varios computadores.',
        'El catálogo de tipos de incidente se construyó a partir de los datos porque no se encontró uno oficial.',
        'Las proyecciones a uno y dos años se basan en datos repetidos, no en datos reales de esos años.',
        'No se elaboraron el video ni las diapositivas del mini-pitch, que quedaron fuera del alcance acordado.',
    ])

    # ------------------------------------------------------------------ Preguntas
    d.h1('Preguntas frecuentes en la sustentación')
    preguntas = [
        ('¿Por qué PySpark fue más lento si es una herramienta de Big Data? ',
         'Porque con pocos datos domina el tiempo de arranque y de organización del trabajo. Su ventaja aparece con '
         'volúmenes grandes y varios computadores.'),
        ('¿Por qué no se eligió Polars, si obtuvo el mejor puntaje? ',
         'Porque la actividad exige PySpark y porque Polars trabaja en un solo computador; se recomienda como '
         'herramienta complementaria.'),
        ('¿Se borró algún dato? ',
         'Solo las copias exactas: 44 filas idénticas en todos sus datos, que se guardaron aparte. Los vacíos '
         'y la fila sin localidad se conservaron, marcados con una nota de calidad.'),
        ('¿Cómo se sabe que las tres herramientas hicieron lo mismo? ',
         'Se compararon sus resultados fila por fila con una huella digital del contenido, y coincidieron.'),
        ('¿Por qué Parquet y no CSV? ',
         'Porque ocupa menos espacio, guarda el tipo de cada dato y permite leer solo lo necesario.'),
        ('¿Qué falta para llevarlo a producción? ',
         'Un catálogo oficial de códigos, la carga automática de cada mes nuevo y ejecutarlo en un clúster con '
         'datos de al menos un año.'),
    ]
    for pregunta, respuesta in preguntas:
        d.p(respuesta, pregunta)

    # ------------------------------------------------------------------ Glosario
    d.h1('Glosario')
    d.tabla(['Término', 'Significado'], [
        ['Benchmark', 'Prueba de rendimiento que compara herramientas haciendo la misma tarea.'],
        ['Clúster', 'Grupo de computadores que trabajan juntos como si fueran uno.'],
        ['JVM', 'Máquina virtual de Java: el entorno sobre el que corre Spark; tarda unos segundos en arrancar.'],
        ['Partición', 'Cada parte en que se divide un conjunto de datos para guardarlo o procesarlo.'],
        ['Pipeline', 'Secuencia automática de pasos que transforma los datos de entrada en resultados.'],
        ['Throughput', 'Cantidad de registros procesados por unidad de tiempo.'],
        ['Validación', 'Comprobación automática de que los datos cumplen una regla de calidad.'],
        ['Nulo', 'Celda sin valor: el dato no existe o no se registró.'],
    ], [4, 12], titulo='Términos técnicos utilizados')
    d.referencias([refs[c] for c in ('guia', 'datos', 'spark', 'parquet', 'colab', 'zaharia')], tam=9.5)
    return d.guardar(ruta)


def guion_sustentacion(R, matriz, ruta, datos):
    """Hoja de sustentación de 5 minutos (1 + 2 + 1,5 + 0,5, distribución de la guía) con las cifras medidas."""
    k = _datos_clave(R, matriz)
    e2e, cal, ent, ps, pp, poc = k['e2e'], k['cal'], k['ent'], k['ps'], k['pp'], k['poc']
    inv = R['inventario']
    rapida = k['rapida']
    u = poc['ultima_spark']
    var = poc['variante'].iloc[0] if len(poc['variante']) else None
    orden = sorted(matriz['ponderado'], key=matriz['ponderado'].get, reverse=True)
    esc = R.get('escala')
    obs = u.get('observaciones_por_motivo') or {}
    L = [
        '# Hoja de sustentación — 5 minutos', '',
        'Distribución de la guía (referencial): 1 min introducción · 2 min arquitectura · 1,5 min demo del PoC · '
        '0,5 min conclusiones. Todas las cifras salen de la ejecución en Colab del ' + fecha_es(ent['fecha']) +
        ' (`resultados/mediciones/`). Si el notebook se vuelve a ejecutar, estas cifras se regeneran.', '',
        '## 1. Introducción (1 min)', '',
        f"- **Problema:** procesar las llamadas de emergencia de la Línea 123 de Bogotá (Secretaría Distrital de Salud), "
        f"julio–septiembre de 2026: {f(inv['total_registros'], 0)} registros en 3 CSV (~3 MB).",
        '- **Qué se hizo:** (1) benchmark de Pandas, Polars y PySpark con la misma lógica; (2) matriz de decisión; '
        '(3) arquitectura y PoC en PySpark sobre Google Colab, con los datos en Drive.',
        '- **Decisión de entrada:** PySpark es la herramienta principal porque la actividad lo exige y porque es la única '
        'de las tres que escala a varios computadores.', '',
        '## 2. Arquitectura y decisiones (2 min) — mostrar `diagrama_arquitectura.png`', '',
        '- **Cuatro capas:** ingesta (CSV en Drive, SHA-256 verificado) → procesamiento (PySpark en modo local) → '
        'almacenamiento (Parquet particionado por periodo en Drive) → salidas (indicadores e informes).',
        '- **Calidad:** esquema explícito, nulos sin imputar, solo se eliminan las copias duplicadas exactas '
        f"({f(ps['rechazados'], 0)} filas, guardadas aparte); lo demás se conserva con una observación de calidad.",
        '- **Optimización:** broadcast join del catálogo (23 filas), partición por periodo y Parquet. '
        + (f"Por periodo salen {int(u['parquet_archivos'])} archivos; por periodo y localidad saldrían "
           f"{int(var['parquet_archivos'])}, de ~{f(var['parquet_bytes'] / var['parquet_archivos'] / 1024, 0)} KB cada uno "
           "(problema de archivos pequeños)." if var is not None else ''),
        '- **Matriz (30/25/20/15/10):** ' + ', '.join(f"{NOMBRE[h]} {f(matriz['ponderado'][h], 2)}" for h in orden) +
        '. Gana la más rápida en este volumen, pero se mantiene PySpark por exigencia y escalabilidad.', '',
        '## 3. Demo del PoC (1,5 min) — notebook con celdas pre-ejecutadas', '',
        '1. Sección 4.2: las 3 ejecuciones de Spark y de Pandas (tiempos por ejecución).',
        '2. Sección 4.3: tabla de métricas y promedios.',
        '3. Sección 4.4: log de validación; mostrar los controles V01–V20.',
        '4. Sección 4.5: el Parquet releído en Drive (`periodo=2026-07/08/09`).',
        '',
        '| Métrica (media de 3 ejecuciones) | PySpark | Pandas |', '|---|---:|---:|',
        f"| Tiempo total (s) | {f(ps['tiempo_total_s'], 2)} | {f(pp['tiempo_total_s'], 2)} |",
        f"| Memoria del sistema, pico (%) | {f(ps['mem_sis_pico_pct'], 1)} | {f(pp['mem_sis_pico_pct'], 1)} |",
        f"| RSS pico proceso + JVM (MB) | {f(ps['rss_pico_mb'], 0)} | {f(pp['rss_pico_mb'], 0)} |",
        f"| Throughput (registros/min) | {f(ps['throughput_reg_min'], 0)} | {f(pp['throughput_reg_min'], 0)} |",
        f"| Tasa de éxito de validaciones (%) | {f(ps['tasa_validaciones_pct'], 1)} | {f(pp['tasa_validaciones_pct'], 1)} |",
        '', '## 4. Conclusiones (0,5 min)', '',
        f"- Con 3 meses de datos, **Pandas es {f(veces(ps['tiempo_total_s'], pp['tiempo_total_s']), 0)} veces más rápido "
        'que PySpark** y usa menos memoria: el arranque de la JVM y la coordinación pesan más que el procesamiento. '
        'No se presenta una mejora que no existió.',
        '- La justificación de PySpark es la escalabilidad horizontal; la proyección a 1–2 años está en '
        '`analisis_escalabilidad.docx` y es una **estimación**, no una medición.',
        '- Siguientes pasos: catálogo oficial de códigos, carga incremental mensual y ejecución en un clúster.', '',
        '## Preguntas probables y respuestas cortas', '',
        '- **¿Por qué PySpark fue más lento?** Costes fijos (JVM, planificación, serialización) que con 3 MB dominan. '
        + (f"En una segunda pasada con el motor encendido bajó de {f(e2e.loc['spark', 'seg'], 1)} s a "
           f"{f(cal.loc['spark', 'seg'], 1)} s." if cal is not None else ''),
        f"- **¿Por qué no Polars si tuvo el mejor puntaje?** La actividad exige PySpark y Polars trabaja en un solo nodo; "
        f"queda como complementaria. La más rápida fue {NOMBRE[rapida]}.",
        '- **¿Se borró algún dato?** Solo las copias idénticas en el 100 % de sus atributos, que quedan en '
        '`duplicados_eliminados/`. Nulos y anomalías se conservan marcados en `observacion_calidad`'
        + (f" ({', '.join(f'{m}: {c}' for m, c in obs.items())})." if obs else '.'),
        '- **¿Cómo sé que las tres herramientas hicieron lo mismo?** Se compararon filas, esquema, conteo por tipo y el '
        'SHA-256 del contenido canónico (sección 3.4 del notebook).',
        '- **¿Son medidas confiables?** Cada repetición corre en un proceso nuevo; 1 calentamiento descartado y '
        f"{R['bench']['repeticiones']} repeticiones, con el orden de herramientas rotado.",
        '- **¿Es un clúster?** No: Spark corre en modo local dentro de una VM de Colab (2 vCPU). Un clúster '
        'cambiaría `spark.master` y los parámetros de executors, no el código.',
        '- **¿De dónde sale el catálogo de tipos?** Se derivó de los datos; **no es oficial** y así se rotula.',
        '- **¿Y las proyecciones?** Se basan en datos replicados 4× y 8×, rotulados como sintéticos.', '',
        '## Qué tener a la mano', '',
        '- `Actividad_2/diagrama_arquitectura.png` · `comparativo_poc_vs_pandas.png` · `metricas_poc.xlsx`',
        '- `U2_procesamiento_distribuido.ipynb` ya ejecutado (secciones 3 y 4)',
        '- `Sustentacion/Sustentacion_proyecto_U2.docx` (explicación para no especialistas)', '',
    ]
    ruta = __import__('pathlib').Path(ruta)
    ruta.write_text('\n'.join(L), encoding='utf-8')
    return ruta

**Verificación con LibreOffice (`verificacion.py`).** Recalcula las fórmulas de los xlsx y busca errores;
exporta los docx a PDF para contar sus páginas.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/verificacion.py
"""Verificación de entregables en Colab: recálculo de fórmulas xlsx y conteo de páginas de docx.

Usa LibreOffice en modo headless, instalado en el runtime de Colab (notebook de informes:
`apt-get install libreoffice-calc libreoffice-writer`). Las celdas que escribe openpyxl no traen valor en caché:
LibreOffice las calcula al convertir el libro y así se detectan errores de fórmula. Los docx se exportan a PDF
para contar las páginas tal como se renderizan.
"""
import shutil
import subprocess
import tempfile
from pathlib import Path

from openpyxl import load_workbook

ERRORES = ('#DIV/0!', '#REF!', '#NAME?', '#VALUE!', '#N/A', '#NUM!', '#NULL!', 'Err:502', 'Err:504')


def _soffice(*args, salida):
    binario = shutil.which('soffice') or shutil.which('libreoffice')
    if not binario:
        raise RuntimeError('Falta LibreOffice en el runtime: ejecute la celda de instalación del notebook de informes.')
    perfil = Path(tempfile.mkdtemp(prefix='lo_perfil_'))  # perfil aislado: evita bloqueos entre ejecuciones
    subprocess.run([binario, f'-env:UserInstallation=file://{perfil}', '--headless', '--norestore', *args,
                    '--outdir', str(salida)], check=True, capture_output=True, timeout=300)


def libro_calculado(ruta):
    """Copia del libro con los valores calculados por LibreOffice (para leer resultados; no se entrega)."""
    ruta = Path(ruta).resolve()
    tmp = Path(tempfile.mkdtemp(prefix='recalc_'))
    _soffice('--convert-to', 'xlsx:Calc MS Excel 2007 XML', str(ruta), salida=tmp)
    return tmp / ruta.name


def recalcular_excel(ruta):
    """Recalcula con LibreOffice y cuenta fórmulas, errores y fórmulas sin resultado.

    El libro entregado conserva sus fórmulas; Excel, LibreOffice o Google Sheets las recalculan al abrirlo.
    """
    ruta = Path(ruta).resolve()
    calculado = libro_calculado(ruta)
    original, valores = load_workbook(ruta), load_workbook(calculado, data_only=True)
    formulas, errores, vacias = 0, [], 0
    for ws in original.worksheets:
        for fila in ws.iter_rows():
            for c in fila:
                if isinstance(c.value, str) and c.value.startswith('='):
                    formulas += 1
                    v = valores[ws.title][c.coordinate].value
                    if isinstance(v, str) and v in ERRORES:
                        errores.append(f'{ws.title}!{c.coordinate}={v}')
                    elif v is None:
                        vacias += 1
    shutil.rmtree(calculado.parent, ignore_errors=True)
    return {'archivo': ruta.name, 'formulas': formulas, 'errores': errores, 'formulas_sin_valor': vacias,
            'motor': 'LibreOffice (Colab)'}


def docx_a_pdf(ruta_docx, ruta_pdf):
    """Exporta a PDF con LibreOffice y devuelve el número de páginas renderizadas."""
    import pymupdf

    ruta_docx, ruta_pdf = Path(ruta_docx).resolve(), Path(ruta_pdf).resolve()
    ruta_pdf.parent.mkdir(parents=True, exist_ok=True)
    _soffice('--convert-to', 'pdf', str(ruta_docx), salida=ruta_pdf.parent)
    generado = ruta_pdf.parent / (ruta_docx.stem + '.pdf')
    if generado != ruta_pdf:
        generado.replace(ruta_pdf)
    with pymupdf.open(ruta_pdf) as pdf:
        return pdf.page_count

**Orquestación (`generador.py`)**: genera y verifica todos los informes.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/generador.py
"""Genera y verifica todos los informes a partir de las mediciones guardadas (sin volver a medir).

En Colab (notebook de informes): generar_todo(cfg.dir_resultados, cfg.dir_informes) escribe en Drive los xlsx, docx, png y drawio,
los verifica con LibreOffice (fórmulas sin errores, páginas de cada docx) y guarda verificacion_informes.json.
"""
import json
from pathlib import Path

from openpyxl import load_workbook

from . import diagrama, documentos, excel, graficos, sustentacion
from .resumen import cargar_todo
from .verificacion import docx_a_pdf, libro_calculado, recalcular_excel


def leer_matriz(ruta_xlsx, ruta_recalculada=None):
    """Lee la matriz de un libro con valores calculados (el recalculado por LibreOffice/Excel)."""
    ws = load_workbook(ruta_recalculada or ruta_xlsx, data_only=True)['Matriz']
    filas = {ws.cell(i, 1).value: i for i in range(1, ws.max_row + 1) if ws.cell(i, 1).value}
    herramientas = {'pandas': 3, 'polars': 4, 'spark': 5}
    criterios = ['Rendimiento', 'Facilidad de uso', 'Escalabilidad', 'Recursos', 'Curva de aprendizaje']
    return {
        'pesos': {c: ws.cell(filas[c], 2).value for c in criterios},
        'puntajes': {c: {h: ws.cell(filas[c], col).value for h, col in herramientas.items()} for c in criterios},
        'ponderado': {h: ws.cell(filas['Puntaje ponderado'], col).value for h, col in herramientas.items()},
        'loc': {h: ws.cell(filas['LOC totales'], col).value for h, col in herramientas.items()},
    }


def generar_todo(dir_resultados, dir_informes, dir_render=None, datos_doc=None, dir_base=None):
    dir_resultados, dir_informes = Path(dir_resultados), Path(dir_informes)
    if datos_doc is None:
        from ..config import cargar_config
        datos_doc = cargar_config().documentos
    dir_informes.mkdir(parents=True, exist_ok=True)
    dir_render = Path(dir_render or dir_informes / 'verificacion_pdf')
    dir_render.mkdir(parents=True, exist_ok=True)
    R = cargar_todo(dir_resultados)
    verif = {}

    rutas = {
        'g_operaciones': graficos.grafico_operaciones(R['bench']['med'], dir_informes / 'benchmark_operaciones.png'),
        'g_totales': graficos.grafico_totales(R['bench']['med'], dir_informes / 'benchmark_totales_memoria.png'),
        'g_totales_doc': graficos.grafico_totales(R['bench']['med'], dir_informes / 'benchmark_totales_memoria_2filas.png',
                                                  por_fila=3),
        'g_poc': graficos.grafico_poc(R['poc']['principal'], dir_informes / 'comparativo_poc_vs_pandas.png'),
        'diagrama': diagrama.escribir_png(dir_informes / 'diagrama_arquitectura.png'),
    }
    diagrama.escribir_drawio(dir_informes / 'diagrama_arquitectura.drawio')

    # Las líneas de código las cuenta el notebook de análisis (sección del benchmark) y las deja en mediciones/.
    loc = json.loads((dir_resultados / 'mediciones' / 'loc_benchmark.json').read_text(encoding='utf-8'))
    xb = excel.reporte_benchmark(R['bench']['crudo'], loc, R['bench']['equivalencia'], R['entorno'],
                                 dir_informes / 'reporte_comparativo_benchmark.xlsx')
    verif['reporte_comparativo_benchmark.xlsx'] = recalcular_excel(xb)
    crudo_ok = R['poc']['crudo'][R['poc']['crudo']['estado'] == 'ok']
    xp = excel.reporte_poc(crudo_ok, R['poc']['ultima_spark']['controles'], dir_informes / 'metricas_poc.xlsx', R['entorno'])
    verif['metricas_poc.xlsx'] = recalcular_excel(xp)
    matriz = leer_matriz(xb, libro_calculado(xb))

    config_spark = crudo_ok[crudo_ok['herramienta'] == 'spark']['config_spark'].dropna().iloc[-1]
    docs = {
        'analisis_decision_tecnologica.docx': documentos.doc_decision(R, matriz, rutas, dir_informes / 'analisis_decision_tecnologica.docx', datos_doc),
        'arquitectura_tecnica.docx': documentos.doc_arquitectura(R, matriz, rutas, config_spark, dir_informes / 'arquitectura_tecnica.docx', datos_doc),
        'analisis_escalabilidad.docx': documentos.doc_escalabilidad(R, dir_informes / 'analisis_escalabilidad.docx', datos_doc),
    }
    carpeta_sust = Path(dir_base or dir_informes.parent) / 'Sustentacion'
    carpeta_sust.mkdir(parents=True, exist_ok=True)
    docs['Sustentacion_proyecto_U2.docx'] = sustentacion.doc_sustentacion(
        R, matriz, rutas, carpeta_sust / 'Sustentacion_proyecto_U2.docx', datos_doc)
    sustentacion.guion_sustentacion(R, matriz, carpeta_sust / 'Hoja_de_sustentacion_5min.md', datos_doc)
    for nombre, ruta in docs.items():
        verif[nombre] = {'paginas': docx_a_pdf(ruta, dir_render / nombre.replace('.docx', '.pdf'))}
    verif['matriz'] = matriz
    (dir_resultados / 'mediciones' / 'verificacion_informes.json').write_text(
        json.dumps(verif, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    return verif

**Organización de la carpeta del proyecto (`entregables.py`).** Copia cada informe a la carpeta de su actividad
(sin duplicados), escribe `QUE_SE_PIDE.md` en cada una y `LEEME.md` en la raíz.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/entregables.py
"""Organiza la ÚNICA carpeta del proyecto en Drive (MyDrive/BigData_U2) con lo que pide la guía U2.

Decisiones del usuario (2026-11-10): los programas son SOLO dos notebooks, y en Drive hay una sola carpeta con lo
necesario (datos, informes, notebooks, evidencia), sin copias duplicadas. Estructura final:

    BigData_U2/
      LEEME.md                            índice tarea → archivo y cómo ejecutar
      U2_procesamiento_distribuido.ipynb  datos, benchmark, PoC, validaciones (con sus salidas)
      U2_informes_y_entregables.ipynb     genera y verifica los informes y organiza esta carpeta
      datos/                              los 3 CSV (SHA-256 verificado)
      Actividad_1/  Actividad_2/          informes de cada actividad (QUE_SE_PIDE.md en cada una)
      Sustentacion/                       explicación para no especialistas
      resultados/                         evidencia: mediciones, logs y Parquet del PoC

Los informes se generan en una carpeta temporal del VM (`origen_informes`) y aquí se copian a su actividad.
Cada archivo va en UNA sola actividad; si otra tarea lo usa, QUE_SE_PIDE.md lo referencia con su ruta.
"""
import shutil
from pathlib import Path

DRIVE = 'MyDrive/BigData_U2'
NB_ANALISIS = 'U2_procesamiento_distribuido.ipynb'
NB_INFORMES = 'U2_informes_y_entregables.ipynb'
ACT1, ACT2 = 'Actividad_1', 'Actividad_2'

ACTIVIDADES = {
    ACT1: {
        'titulo': 'Actividad 1 · Análisis comparativo de herramientas (formativa)',
        'archivos': {  # nombre en la actividad -> origen (relativo a la carpeta temporal de informes o a la base)
            'reporte_comparativo_benchmark.xlsx': 'reporte_comparativo_benchmark.xlsx',
            'analisis_decision_tecnologica.docx': 'analisis_decision_tecnologica.docx',
            'benchmark_operaciones.png': 'benchmark_operaciones.png',
            'benchmark_totales_memoria.png': 'benchmark_totales_memoria.png',
        },
        'tareas': [
            ('1.1 Benchmark comparativo de herramientas',
             'Comparación empírica Pandas vs Polars vs PySpark con los mismos 3 archivos mensuales; measure_performance() '
             'con tiempo y memoria (psutil); carga, filtrado USAQUEN/CHAPINERO, agregación por tipo, join con la tabla de '
             'códigos y exportación Parquet; tiempo total y por operación; líneas de código por operación.',
             [f'../{NB_ANALISIS} (parte 3: benchmark con measure_performance, con sus salidas)',
              'reporte_comparativo_benchmark.xlsx (hojas Mediciones, Promedios, LOC, Graficos, Equivalencia)',
              'benchmark_operaciones.png', 'benchmark_totales_memoria.png',
              '../resultados/mediciones/benchmark.jsonl (mediciones crudas)',
              '../resultados/mediciones/equivalencia_benchmark.json']),
            ('1.2 Matriz de decisión tecnológica',
             'Criterios y pesos: rendimiento 30 %, facilidad de uso 25 %, escalabilidad 20 %, recursos 15 %, curva de '
             'aprendizaje 10 %; puntuación 1–5 basada en el benchmark; puntaje ponderado; restricciones de Colab.',
             ['reporte_comparativo_benchmark.xlsx (hojas Reglas y Matriz, con fórmulas SUMPRODUCT)',
              'analisis_decision_tecnologica.docx (§4)']),
            ('1.3 Selección tecnológica justificada',
             'Herramienta principal (obligatoria: PySpark) y complementarias, criterios de uso por volumen, plan de '
             'implementación por fases y limitaciones.',
             ['analisis_decision_tecnologica.docx (§5–§9)', 'reporte_comparativo_benchmark.xlsx (hoja Matriz)']),
        ],
        'nota': 'Los puntajes de la matriz ponderada están en reporte_comparativo_benchmark.xlsx (hoja Matriz) y su '
                'interpretación, en analisis_decision_tecnologica.docx. PySpark se mantiene como herramienta principal por '
                'exigencia de la actividad y por su escalabilidad horizontal, aunque con este volumen no sea la más rápida.',
    },
    ACT2: {
        'titulo': 'Actividad 2 · Diseño de arquitectura y PoC en PySpark (evaluativa)',
        'archivos': {
            'arquitectura_tecnica.docx': 'arquitectura_tecnica.docx',
            'diagrama_arquitectura.png': 'diagrama_arquitectura.png',
            'diagrama_arquitectura.drawio': 'diagrama_arquitectura.drawio',
            'metricas_poc.xlsx': 'metricas_poc.xlsx',
            'comparativo_poc_vs_pandas.png': 'comparativo_poc_vs_pandas.png',
            'analisis_escalabilidad.docx': 'analisis_escalabilidad.docx',
            'validacion_poc_spark.txt': 'resultados/logs/validacion_poc_spark.txt',
        },
        'tareas': [
            ('2.1 Documento de arquitectura técnica',
             'Documento .docx de 5 a 8 páginas con diagrama (ingesta, procesamiento, almacenamiento, salida), decisiones '
             'justificadas, configuraciones de PySpark, particionamiento mes/localidad, métricas de monitoreo/KPIs, '
             'estimación de recursos, riesgos; diagrama .png de alta resolución exportado aparte.',
             ['arquitectura_tecnica.docx (5–8 páginas, verificado al exportar a PDF)', 'diagrama_arquitectura.png (300 dpi)',
              'diagrama_arquitectura.drawio (fuente editable)']),
            ('2.2 Proof of Concept en PySpark',
             'Notebook de Colab funcional y documentado con outputs; PySpark en Colab; ingesta de 3 CSV desde Google Drive; '
             'limpieza de nulos, tipos, estandarización de códigos; validación de calidad; exportación Parquet subida a '
             'Drive; log de validación .txt (registros procesados, errores, tiempo).',
             [f'../{NB_ANALISIS} (partes 4 a 6: PoC en PySpark, con sus salidas)',
              f'../resultados/parquet/poc/spark/e1..e3/ (Parquet en Google Drive: {DRIVE}/resultados/parquet/poc/spark/)',
              'validacion_poc_spark.txt (log de validación)', '../datos/ (los 3 CSV de entrada)']),
            ('2.3 Mini-pitch técnico', 'Presentación de máx. 5 minutos (6 slides) y video .mp4.',
             ['EXCLUIDO del alcance del encargo. Insumos disponibles: diagrama_arquitectura.png, '
              'comparativo_poc_vs_pandas.png y ../Sustentacion/Sustentacion_proyecto_U2.docx']),
            ('2.4 Métricas de rendimiento',
             'PoC ejecutado 3 veces: tiempo total (s), memoria pico (%), throughput (registros/min), tasa de éxito de '
             'validaciones (%); comparación con las métricas de Pandas del benchmark; reporte .xlsx con 3 ejecuciones y '
             'promedios; gráfico .png de tiempo y memoria vs Pandas; análisis de escalabilidad .docx de 1 página.',
             ['metricas_poc.xlsx', 'comparativo_poc_vs_pandas.png', 'analisis_escalabilidad.docx (1 página)',
              '../resultados/mediciones/poc.jsonl y escalabilidad.jsonl (mediciones crudas)',
              '../Actividad_1/reporte_comparativo_benchmark.xlsx (métricas de Pandas del benchmark inicial)',
              'arquitectura_tecnica.docx (§9 recursos y §10 KPIs)']),
        ],
        'nota': 'Las métricas de las 3 ejecuciones y sus promedios están en metricas_poc.xlsx, y el detalle de los controles, '
                'en validacion_poc_spark.txt. Con 3 meses de datos PySpark no mejora a Pandas en tiempo ni en memoria, y así '
                'se reporta. Solo se eliminan las copias duplicadas exactas (100 % de sus atributos iguales), que quedan '
                'registradas aparte; los nulos y demás anomalías se conservan marcados en observacion_calidad.',
    },
}
EXCLUIDOS = 'Video, presentación/diapositivas, guion del mini-pitch y capturas manuales: excluidos por alcance del encargo.'
ESTRUCTURA = [
    ('LEEME.md', 'este índice'),
    ('Guia de actividades_U2 (6) (1).pdf', 'la guía de actividades de la Unidad 2'),
    (NB_ANALISIS, 'todo el procesamiento, narrado sección por sección y con sus salidas de Colab'),
    (NB_INFORMES, 'genera y verifica los informes (LibreOffice en Colab) y organiza esta carpeta'),
    ('datos/', 'los 3 CSV de julio, agosto y septiembre de 2026 (huella SHA-256 verificada)'),
    (f'{ACT1}/', 'entregables de la Actividad 1 y QUE_SE_PIDE.md'),
    (f'{ACT2}/', 'entregables de la Actividad 2 y QUE_SE_PIDE.md'),
    ('Sustentacion/', 'documento que explica todo el proyecto a una persona sin conocimientos técnicos y '
                      'hoja de sustentación de 5 minutos (Hoja_de_sustentacion_5min.md)'),
    ('resultados/', 'evidencia: mediciones crudas (JSONL/JSON), logs de validación y Parquet del PoC'),
]


def _resolver(origen, raices):
    for raiz in raices:
        p = Path(raiz) / origen
        if p.exists():
            return p
    return None


def _copiar(origen, destino):
    destino.parent.mkdir(parents=True, exist_ok=True)
    if origen.is_dir():
        shutil.copytree(origen, destino, ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '_SUCCESS', '.*.crc'))
    else:
        shutil.copy2(origen, destino)


def construir(base, origen_informes):
    """Rehace las carpetas de actividad dentro de `base` y escribe LEEME.md. No toca datos, notebooks ni resultados."""
    base, raices = Path(base), [Path(origen_informes), Path(base)]
    estados = {}
    indice = ['# Proyecto U2 — Procesamiento distribuido (Emergencias 123 Bogotá)', '',
              'Carpeta única del proyecto en Google Drive con lo que pide la guía de la Unidad 2. Los programas son '
              'únicamente los dos notebooks de esta carpeta.', '', '## Contenido', '']
    indice += [f'- `{n}`: {d}' for n, d in ESTRUCTURA]
    indice += ['', '## Cómo ejecutar (solo Google Colab)', '',
               f'1. Subir `{NB_ANALISIS}` y `{NB_INFORMES}` a `{DRIVE}/` en Google Drive.',
               f'2. Abrir `{NB_ANALISIS}` en Colab (runtime CPU) y elegir *Entorno de ejecución → Ejecutar todas*. Colab '
               'pide autorizar Drive. Los CSV se toman de `MyDrive/BigData_Emergencias_123` (carpeta de la Unidad 1) o, si '
               'no están, se descargan del portal Datos Abiertos Bogotá; en ambos casos se verifica su SHA-256.',
               f'3. Ese mismo notebook ejecuta al final `{NB_INFORMES}` (sección 9): genera los informes, los verifica y '
               'organiza esta carpeta. No hay que ejecutar nada más.',
               '']
    for carpeta, act in ACTIVIDADES.items():
        d = base / carpeta
        if d.exists():  # carpeta generada: se rehace para no dejar restos de una corrida anterior
            shutil.rmtree(d)
        d.mkdir(parents=True)
        faltantes = []
        for nombre, origen in act['archivos'].items():
            o = _resolver(origen, raices)
            if o:
                _copiar(o, d / nombre)
            else:
                faltantes.append(origen)
        texto = [f"# {act['titulo']}", '', act['nota'], '', '## Tareas de la guía y archivos que las cumplen', '',
                 'Las rutas que empiezan por `../` están en la carpeta del proyecto (no se duplican aquí).', '']
        for tarea, pedido, archivos in act['tareas']:
            texto += [f'### {tarea}', '', f'**Qué se pide:** {pedido}', '', '**Archivos:**', '']
            texto += [f'- {a}' for a in archivos] + ['']
        estado = 'COMPLETO' if not faltantes else 'INCOMPLETO: falta ' + ', '.join(faltantes)
        texto += [f'**Estado:** {estado}', '', f'_{EXCLUIDOS}_', '']
        (d / 'QUE_SE_PIDE.md').write_text('\n'.join(texto), encoding='utf-8')
        indice += [f"## [{act['titulo']}]({carpeta}/QUE_SE_PIDE.md)", '']
        indice += [f'- {t[0]}' for t in act['tareas']] + ['', f'Estado: {estado}', '']
        estados[carpeta] = estado
    indice += [f'_{EXCLUIDOS}_', '']
    (base / 'LEEME.md').write_text('\n'.join(indice), encoding='utf-8')
    return estados

## 1. Configuración

Las rutas y los parámetros salen de `config/config.yaml` (Anexo A).

In [ ]:
import inspect
from IPython.display import Markdown, display
from emergencias123.config import cargar_config

def ver_codigo(*objetos):
    """Muestra en pantalla el código fuente de funciones o módulos que se están explicando."""
    display(Markdown('```python\n' + '\n\n'.join(inspect.getsource(o) for o in objetos) + '\n```'))

cfg = cargar_config()
print('Entorno:', cfg.entorno, '| datos:', cfg.dir_datos, '| resultados:', cfg.dir_resultados)
faltan = [n for n in ('benchmark.jsonl', 'poc.jsonl', 'escalabilidad.jsonl', 'loc_benchmark.json', 'entorno_colab.json')
          if not (cfg.dir_resultados / 'mediciones' / n).exists()]
assert not faltan, f'Primero ejecuta U2_procesamiento_distribuido.ipynb: faltan las mediciones {faltan}'

## Datos de autoría (completa estos tres campos antes de ejecutar)

Aparecen en el bloque de título de todos los documentos Word (formato APA 7). Si se dejan entre corchetes, los
documentos salen con el marcador `[...]` para completarlos a mano.

In [ ]:
# Si este notebook se ejecuta desde la sección 9 del de procesamiento, esos valores ya vienen definidos allí.
AUTOR = globals().get('AUTOR', '[Nombre del estudiante o del equipo]')
INSTITUCION = globals().get('INSTITUCION', '[Nombre de la institución]')
DOCENTE = globals().get('DOCENTE', '[Nombre del docente]')
datos_doc = {**cfg.documentos, 'autor': AUTOR, 'institucion': INSTITUCION, 'docente': DOCENTE}

## 2. Herramientas de ofimática en el runtime de Colab (LibreOffice, python-docx, PyMuPDF)

In [ ]:
!apt-get update -qq > /dev/null && apt-get install -y -qq libreoffice-calc libreoffice-writer fonts-liberation > /dev/null
%pip install -q python-docx pymupdf
import shutil, importlib.metadata as md
print('LibreOffice:', shutil.which('soffice'), '| python-docx', md.version('python-docx'), '| pymupdf', md.version('pymupdf'))

## 3. Generación y verificación de los informes

In [ ]:
import pandas as pd
from emergencias123.reportes.generador import generar_todo
shutil.rmtree(cfg.dir_informes, ignore_errors=True)
verif = generar_todo(cfg.dir_resultados, cfg.dir_informes, datos_doc=datos_doc, dir_base=cfg.dir_base)
filas = []
for nombre, v in verif.items():
    if nombre.endswith('.xlsx'):
        filas.append({'archivo': nombre, 'verificación': f"{v['formulas']} fórmulas, {len(v['errores'])} errores, "
                      f"{v['formulas_sin_valor']} sin valor ({v['motor']})"})
    elif nombre.endswith('.docx'):
        filas.append({'archivo': nombre, 'verificación': f"{v['paginas']} página(s) al exportar a PDF"})
assert all(not v['errores'] for k, v in verif.items() if k.endswith('.xlsx')), 'Hay fórmulas con error'
assert 5 <= verif['arquitectura_tecnica.docx']['paginas'] <= 8, 'La arquitectura debe tener 5-8 páginas'
assert verif['analisis_escalabilidad.docx']['paginas'] == 1, 'La escalabilidad debe ocupar 1 página'
print('Matriz ponderada:', {h: round(x, 2) for h, x in verif['matriz']['ponderado'].items()})
pd.DataFrame(filas)

### Matriz de decisión tecnológica (tarea 1.2)

Pesos: rendimiento 30 %, facilidad de uso 25 %, escalabilidad 20 %, recursos 15 %, curva de aprendizaje 10 %. Los
puntajes de rendimiento y recursos salen de una regla fija sobre las mediciones; los demás se justifican en
`analisis_decision_tecnologica.docx`. En el Excel el puntaje ponderado es una fórmula `SUMPRODUCT`.

In [ ]:
matriz = verif['matriz']
punt = pd.DataFrame(matriz['puntajes']).T
punt.insert(0, 'peso', pd.Series(matriz['pesos']))
punt.loc['PUNTAJE PONDERADO'] = [None, *[matriz['ponderado'][h] for h in punt.columns[1:]]]
punt.round(2)

In [ ]:
from IPython.display import Image, display
for f in ('benchmark_totales_memoria.png', 'comparativo_poc_vs_pandas.png', 'diagrama_arquitectura.png'):
    display(Image(filename=str(cfg.dir_informes / f), width=1000))

## 4. Carpeta única del proyecto en Drive

Cada informe se copia a la carpeta de su actividad. `QUE_SE_PIDE.md` relaciona cada tarea de la guía con sus
archivos (si una tarea usa un archivo de otra carpeta, se indica su ruta en lugar de duplicarlo) y `LEEME.md`
resume el contenido y cómo ejecutar.

In [ ]:
from emergencias123.reportes.entregables import construir
estados = construir(cfg.dir_base, cfg.dir_informes)
for carpeta, estado in estados.items():
    print(f'{carpeta}: {estado}')
def arbol(ruta, nivel=0, max_nivel=2):
    for p in sorted(ruta.iterdir()):
        if p.name.startswith('.'):
            continue
        print('    ' * nivel + p.name + ('/' if p.is_dir() else ''))
        if p.is_dir() and nivel < max_nivel and p.name not in ('parquet',):
            arbol(p, nivel + 1, max_nivel)
arbol(cfg.dir_base)